# M12 · Prompt Engineering unificado (OpenAI)

Todos los laboratorios de Prompt Engineering clásico con OpenAI (LAB1-9) + Structured Output, fusionados en un único notebook. El coste (prompt caching y Batch API) tiene sus propios notebooks: 04 y 05.

## Índice

1. [Setup común](#setup)
2. [Primeros ejemplos](#lab1)
3. [Iterar](#lab2)
4. [Resumir](#lab3)
5. [Clasificar](#lab4)
6. [Inferir](#lab5)
7. [Transformar](#lab6)
8. [Expandir](#lab7)
9. [Pair programming](#lab8)
10. [Chatbot](#lab9)
11. [**LAB · Structured Output con Pydantic**](#structured-output) · 2026
12. [Coste: notebooks 04 y 05](#coste)


> **Cómo seguir este lab**
>
> Este notebook es el más largo del módulo M12 (150+ celdas). Cubre los 9 LABs clásicos de prompt engineering + un LAB 2026 de Structured Output.
>
> Tres usos posibles:
>
> 1. **Lectura rápida** · pasa por los headers y las celdas markdown. Ve qué tipos de tarea cubre cada LAB. ~25 minutos.
> 2. **Ejecución sin tocar** · pega tu API key, ejecuta todas las celdas. Coste total ~$0.10-0.30 en gpt-4.1-mini. ~40 minutos.
> 3. **Modificar y entender** · cambia los prompts, prueba otros modelos, mide token usage. Aquí se aprende de verdad. Varias horas.
>
> Default model: `gpt-4.1-mini`. Estos LABs enseñan `temperature` y salidas deterministas (`temperature=0`); los reasoning models (`gpt-5-mini`, serie `o*`) **no aceptan `temperature` ni `max_tokens`**. Por eso aquí usamos `gpt-4.1-mini` (chat clásico, barato). El reasoning se ve en el notebook 3. El default del curso (`COURSE_CONVENTIONS.md` §10) es `gpt-5-mini` para labs sin control de sampling.
>
> Los LABs 1-9 son canon Andrew Ng / DeepLearning.ai. El LAB de Structured Output es 2026-developer-essentials; prompt caching y Batch API están en los notebooks 04 y 05.

<a id="setup"></a>
## Setup común

Instalamos dependencias, pedimos la API key una sola vez y definimos los dos helpers `get_completion` y `get_completion_from_messages` que usaremos durante todo el notebook.

In [ ]:
%pip install -qU openai

In [ ]:
import os
import getpass
from openai import OpenAI

# En local la clave se lee de la variable de entorno; en Colab se pide por pantalla.
api_key = os.environ.get("OPENAI_API_KEY") or getpass.getpass("Enter your OpenAI API Key: ")
os.environ["OPENAI_API_KEY"] = api_key
client = OpenAI(api_key=api_key)

DEFAULT_MODEL = "gpt-4.1-mini"  # chat clásico: acepta temperature y max_tokens (gpt-5/o* no)

def get_completion(prompt, model=DEFAULT_MODEL, temperature=0):
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
    )
    return response.choices[0].message.content


def get_completion_from_messages(messages, model=DEFAULT_MODEL, temperature=0):
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
    )
    return response.choices[0].message.content


<a id="lab1"></a>
## LAB 1 · Primeros ejemplos

## Generación de un resumen con GPT
Este fragmento de código prepara un texto sobre física cuántica y formula un prompt para que GPT genere un resumen de este texto en tres puntos. Utiliza la función get_completion definida anteriormente para enviar el prompt a la API de OpenAI e imprime la respuesta obtenida, que sería un resumen conciso del texto introducido.


In [ ]:
text = f"""
La física cuántica es la rama de la física que se encarga \
de fenómenos que ocurren a escala nanométrica, donde las \
leyes de la física clásica dejan de ser aplicables. \
Explora el comportamiento de partículas subatómicas, \
como los electrones y los fotones, y las peculiaridades \
de la superposición y el entrelazamiento cuántico. \
Este campo es fundamental para el desarrollo \
de tecnologías como la criptografía cuántica, los ordenadores \
cuánticos, y ha desafiado nuestra comprensión de la \
realidad a nivel más fundamental.
"""

prompt = f"""
Resume en los tres puntos más importantes el texto delimitado por comillas triples \
```{text}```
"""
response = get_completion(prompt)
print(response)


## Generación de contenido y en resultado JSON
Este código crea un prompt para generar una lista ficticia de tres series relacionadas con "El Señor de los Anillos", incluyendo títulos, autores y géneros. Utiliza la función get_completion con una temperatura de 0.9 para obtener respuestas más creativas. Después, imprime la respuesta generada en formato **JSON**.


In [109]:
prompt = f"""
Inventa una lista de tres series en relación al Señor de los Anillos inventadas junto con \
sus autores y géneros.
Proporcionarlos en formato JSON con las siguientes claves:
book_id, title, author, genre.
"""
response = get_completion(prompt, temperature = 0.9)
print(response)


```json
[
  {
    "book_id": "LOTR001",
    "title": "Las Crónicas de la Tierra Media",
    "author": "Elena Varela",
    "genre": "Fantasía Épica"
  },
  {
    "book_id": "LOTR002",
    "title": "Sombras de Mordor",
    "author": "Carlos Méndez",
    "genre": "Aventura y Fantasía"
  },
  {
    "book_id": "LOTR003",
    "title": "El Legado de los Istari",
    "author": "Marina Gómez",
    "genre": "Fantasía y Misterio"
  }
]
```


## Reescritura de Texto a Formato de Instrucciones Numeradas
Este código transforma un texto que detalla cómo cocinar una paella en una secuencia de instrucciones numeradas. Utiliza la función get_completion para procesar el texto de entrada, reescribiéndolo en un formato de instrucciones paso a paso, y luego imprime el resultado.


In [110]:
text = f"""
¡Cocinar una paella es todo un arte! Aquí tienes los pasos básicos: \
Primero, calienta el aceite en una paellera grande. \
Mientras el aceite se calienta, corta el pollo y el conejo (o las verduras, para una versión vegetariana) y sofríelos hasta que estén dorados. \
Añade el pimiento cortado y continúa sofriendo. \
Después, incorpora el tomate rallado y deja que se cocine un poco. \
Ahora, añade el arroz y rehógalo ligeramente con el resto de los ingredientes. \
Añade el caldo de pollo (o de verduras) y el azafrán, y asegúrate de que el arroz esté bien distribuido. \
Deja que el arroz cocine a fuego medio, sin remover, hasta que absorba el caldo. \
Finalmente, deja reposar unos minutos antes de servir. \
¡Y ya estás listo para disfrutar de una deliciosa paella!
"""

instrucciones = f"""
Se os proporcionará un texto delimitado por comillas triples.
Si contiene una secuencia de instrucciones, \
reescribe esas instrucciones en el siguiente formato:

Paso 1 - ...
Paso 2 - …
…
Paso N - …

\"\"\"{text}\"\"\"
"""

respuesta = get_completion(instrucciones)
print(respuesta)


Paso 1 - Calienta el aceite en una paellera grande.  
Paso 2 - Corta el pollo y el conejo (o las verduras, para una versión vegetariana) y sofríelos hasta que estén dorados.  
Paso 3 - Añade el pimiento cortado y continúa sofriendo.  
Paso 4 - Incorpora el tomate rallado y deja que se cocine un poco.  
Paso 5 - Añade el arroz y rehógalo ligeramente con el resto de los ingredientes.  
Paso 6 - Añade el caldo de pollo (o de verduras) y el azafrán, y asegúrate de que el arroz esté bien distribuido.  
Paso 7 - Deja que el arroz cocine a fuego medio, sin remover, hasta que absorba el caldo.  
Paso 8 - Deja reposar unos minutos antes de servir.


## Ejemplo de Few-Shot Prompting
Este código demuestra el uso de "few-shot prompting", donde se proporcionan ejemplos concretos de interacción (estudiante y profesor) para enseñar al modelo cómo responder a preguntas en un contexto específico, en este caso sobre programación y lenguajes de programación. El objetivo es guiar al modelo para que genere respuestas en un estilo y formato deseados basándose en los ejemplos proporcionados.


In [6]:
prompt = f"""
Tu tarea es responder en un estilo consistente.

<estudiante>: Explícame qué es la programación.

<profesor>: La programación es como construir un rompecabezas; cada pieza de código es una parte que, unida a las otras, crea una solución completa. Comienzas con ideas simples y las combinas para crear soluciones más complejas y poderosas.

<estudiante>: ¿Qué es un lenguaje de programación?

"""
response = get_completion(prompt)
print(response)


<profesor>: Un lenguaje de programación es el conjunto de reglas y símbolos que usamos para comunicarnos con la computadora. Es como un idioma que permite escribir instrucciones claras y precisas para que la máquina entienda qué debe hacer. Cada lenguaje tiene su propia forma de expresar esas instrucciones, pero todos buscan lograr que el programa funcione correctamente.


## Ejemplo de Cadena de Pensamiento
Este código ilustra un ejemplo de "Cadena de Pensamiento" aplicado al análisis de texto. Mediante una serie de acciones secuenciales - resumir un texto, traducir el resumen, identificar nombres propios, y generar un objeto JSON - demuestra cómo descomponer una tarea compleja en pasos más pequeños y manejables.


In [ ]:
text = f"""
A orillas del Abismo de Helm, las fuerzas de Rohan se preparan para la batalla inminente. \
Aragorn, Legolas y Gimli, junto con el Rey Théoden y sus valientes guerreros, revisan las defensas \
y alientan a sus hombres ante el desafío que tienen por delante. El sonido de los martillos y el olor de la antorcha \
llenan el aire mientras fortifican la fortaleza. La tensión crece con la llegada de las sombras de la noche, \
pero también lo hace la esperanza, encendida por la promesa de estandartes que ondearán libres una vez más tras la victoria. \
La determinación de los defensores de Helm es firme, sabiendo que el destino de Rohan pende de esta valiente resistencia."""


tarea = f"""
Realiza las siguientes acciones:
1 - Resume el texto anterior en una frase en el mismo idioma del texto
2 - Traduce el resumen al inglés.
3 - Enumera cada nombre propio en el resumen inglés.
4 - Genera un objeto JSON que contenga las claves siguientes: english_summary, num_names.


Solo responde el json sin el ```json ni el ```

Text:
```{text}```
"""

tarea_2 = f"""
Tu tarea es realizar las siguientes acciones:
1 - Resume el siguiente texto delimitado por
<> con una frase.
2 - Traduce el resumen al francés.
3 - Enumera cada nombre en el resumen francés.
4 - Genera un objeto JSON que contenga las
siguientes claves: french_summary, num_names.

Utiliza el siguiente formato:
Text: <texto para resumir>
Resumen: <resumen>
Traducción: <traducción del resumen>
Nombres: <lista de nombres en el resumen>
Salida JSON: <json con resumen y num_names>

Text: <{text}>
"""
response = get_completion(tarea)
print(response)
print("================")
response = get_completion(tarea_2)
print(response)


## Cadena de pensamiento para corregir: ¿está bien la solución del alumno?

Un alumno ha resuelto un problema de costes. Su planteamiento es correcto, pero **se ha equivocado al sumar**. Primero pedimos al modelo un veredicto directo, sin pasos intermedios, y lo repetimos 5 veces para ver si es estable.


In [ ]:
enunciado = """
Estoy planeando abrir un gimnasio y necesito calcular el coste total del primer año.
- El alquiler del espacio es de 2.000€ al mes
- Las máquinas y el equipo inicial tienen un coste de 50.000€ (pago único)
- Cada mes, tengo que añadir 1.000€ en mantenimiento de las máquinas
- Además, por cada socio hay un coste variable de 10€ al mes por el uso del material deportivo
Si el gimnasio tiene 250 socios, ¿cuál será el coste total del primer año?
"""

solucion_alumno = """
Coste total del primer año: (2.000 * 12) + 50.000 + (1.000 * 12) + (10 * 250 * 12) = 106.000€
"""

prompt = f"""
Determina si la solución del alumno es correcta o no.

Pregunta:
***
{enunciado}
***

Solución del alumno:
***
{solucion_alumno}
***

Responde solo "Correcta" o "Incorrecta".
"""

veredictos = [get_completion(prompt, model="gpt-4.1-nano").strip() for _ in range(5)]
print("Sin pasos (5 ejecuciones):", veredictos)


Fíjate en el veredicto: sin pasos intermedios, el modelo pequeño ve una fórmula bien planteada y la da por buena. Pero la suma está mal: 24.000 + 50.000 + 12.000 + 30.000 = **116.000€**, no 106.000€. En nuestra ejecución respondió «Correcta» las 5 veces.

En la siguiente celda le obligamos a resolver primero el problema, con un hueco explícito para sus pasos y su total, y solo después comparar con el alumno.


In [ ]:
prompt = f"""
Tu tarea es determinar si la solución del alumno es correcta o no.
Para resolver el problema, haz lo siguiente:

- Primero, trabaja tu propia solución al problema incluyendo el total final.
- Después, compara tu total con el del alumno y evalúa si la solución del alumno es correcta o no.
No decidas si la solución del alumno es correcta hasta que no hayas resuelto tú mismo el problema.

Utiliza el siguiente formato:

Tu solución:
***
Pasos y total
***

¿Coincide tu total con el del alumno?
***
Sí o no
***

Veredicto:
***
Correcta o Incorrecta
***

Pregunta:
***
{enunciado}
***

Solución del alumno:
***
{solucion_alumno}
***
"""

respuestas = [get_completion(prompt, model="gpt-4.1-nano") for _ in range(5)]
print(respuestas[0])

veredictos = ["Incorrecta" if "Incorrecta" in r.rsplit("Veredicto", 1)[-1] else "Correcta" for r in respuestas]
print("\nCon pasos (5 ejecuciones):", veredictos)


Mismo modelo (`gpt-4.1-nano`), mismo enunciado, misma solución del alumno: lo único que cambia es que ahora el modelo **calcula antes de juzgar**. Al escribir sus pasos llega a 116.000€, ve que no coincide con 106.000€ y marca la solución como incorrecta. En nuestra ejecución, «Incorrecta» las 5 veces.

> Con modelos de razonamiento (notebook 03) el modelo ya piensa internamente antes de responder, así que pedir los pasos de forma explícita importa menos. Con modelos pequeños y sin razonamiento, como aquí, sigue marcando la diferencia.


## Alucinaciones

Con una premisa absurda, como la de la celda siguiente, los modelos actuales suelen avisar de que es falsa o dejan claro que escriben ficción: nadie se lo va a creer. La alucinación peligrosa es la **verosímil**: la que suena exactamente a lo que esperabas leer.


In [ ]:
prompt = f"""
Escribe la biografía de Aragorn como Jefe de Gobierno de Andorra
"""
respuesta = get_completion(prompt)
print(respuesta)


Ahora una petición verosímil: referencias científicas de nicho con DOI. La respuesta tendrá el formato perfecto. La pregunta es si cada DOI apunta de verdad al artículo que dice el modelo. (A veces el modelo se niega porque no puede verificarlo, que es la respuesta correcta; si pasa, vuelve a ejecutar.)


In [ ]:
prompt = """
Dame 3 artículos científicos publicados en 2021 sobre la suplementación con creatina
y la función cognitiva. Para cada uno: autores, título, revista y DOI.
"""
referencias = get_completion(prompt)
print(referencias)


**Nunca des por buena una cita sin verificarla.** Que un DOI exista no significa que sea el del artículo citado. Lo comprobamos contra Crossref, el registro oficial de DOIs: para cada DOI de la respuesta pedimos su título real y miramos si coincide con lo que ha escrito el modelo. Lo habitual es que alguno no exista y que otro exista pero apunte a un artículo que no tiene nada que ver.


In [ ]:
import re, json, urllib.request, urllib.error

def titulo_real(doi):
    """Título real de un DOI según Crossref. "NO EXISTE" si Crossref no lo conoce, None si no se pudo consultar."""
    peticion = urllib.request.Request(f"https://api.crossref.org/works/{doi}",
                                      headers={"User-Agent": "keepcoding-prompt-lab/1.0"})
    try:
        with urllib.request.urlopen(peticion, timeout=15) as r:
            return (json.load(r)["message"].get("title") or ["(sin título)"])[0]
    except urllib.error.HTTPError as e:
        return "NO EXISTE" if e.code == 404 else None
    except Exception:
        return None

def aparece_en(titulo, texto):
    """¿El título real aparece (al menos el 60% de sus palabras) en lo que escribió el modelo?"""
    palabras = [w for w in re.findall(r"\w+", titulo.lower()) if len(w) > 3]
    return bool(palabras) and sum(w in texto.lower() for w in palabras) / len(palabras) >= 0.6

dois = list(dict.fromkeys(d.rstrip(".") for d in re.findall(r"10\.\d{4,9}/[^\s)\]*,;]+", referencias)))
if not dois:
    print("Esta vez el modelo no ha dado DOIs (a veces se niega). Vuelve a ejecutar la celda anterior.")
for doi in dois:
    real = titulo_real(doi)
    if real is None:
        veredicto = "no se ha podido consultar Crossref (red o límite de peticiones), prueba más tarde"
    elif real == "NO EXISTE":
        veredicto = "EL DOI NO EXISTE"
    elif aparece_en(real, referencias):
        veredicto = "coincide con lo citado"
    else:
        veredicto = "EXISTE, PERO ES OTRO ARTÍCULO"
    print(f"DOI {doi} · {veredicto}")
    if real not in (None, "NO EXISTE"):
        print(f"   Título real en Crossref: {real}")


<a id="lab2"></a>
## LAB 2 · Iterar

## Generar la descripción de un producto a partir de sus datos técnicos


In [11]:
product_description = """

Características principales

Ligereza
Cuadro ligero de aluminio 6061 Serie Hydro.


Precisión
Transmisión monoplato SRAM NX Eagle


Amortiguación
Rockshox Judy Silver TK R 29er 100mm TA15/110 // Matte Black

Información técnica
Cuadro
Fabricado en aluminio de alta calidad, ligero y rígido.
Geometría de carrera mejorada para mayor estabilidad y capacidad de respuesta.

Grupo
Transmisión SRAM NX Eagle de 12 velocidades para cambios de marcha precisos
Biela NX Eagle 175 mm // 32T
Casete SRAM PG-1230 Eagle // 12s // 11-50T
Desviador SRAM NX Eagle // 12s

Ruedas
Ruedas Forza XCD-SL 29", diseñadas para un gran rendimiento en la montaña.
Rueda delantera Forza XCD-SL, 29er // TA 15x110mm // Clincher-TLR // 23mm
Rueda trasera Forza XCD SL // TA12-148 // 23c // TLR

Neumáticos
Vittoria Barzo // 622x55 // Full Black

Frenos
Frenos de disco SRAM Level T DB para una frenada fiable.

Horquilla
Horquilla Rockshox Judy Silver TK R de 100 mm para una conducción suave en terrenos difíciles.

Manillar
4ZA Stratos Flatbar // 740mm

Potencia
Forza Stratos // 90 mm // Black Glossy

Garantía (años)
3

"""

In [122]:
prompt = f"""
Your task is to help a marketing team create a
description for a retail website of a product based
on a technical fact sheet.

Write a product description based on the information
provided in the technical specifications delimited by
triple backticks.

Technical specifications: ```{product_description}```
"""
response = get_completion(prompt)
print(response)

Descubre la bicicleta perfecta para los amantes del mountain bike que buscan ligereza, precisión y rendimiento en cada ruta. Este modelo cuenta con un cuadro de aluminio 6061 Serie Hydro, que combina alta calidad, ligereza y rigidez para ofrecer una geometría de carrera mejorada, garantizando mayor estabilidad y capacidad de respuesta en cualquier terreno.

Equipada con la transmisión monoplato SRAM NX Eagle de 12 velocidades, disfrutarás de cambios de marcha precisos y suaves gracias a su biela de 175 mm y casete 11-50T. Las ruedas Forza XCD-SL de 29" están diseñadas para un rendimiento óptimo en montaña, con neumáticos Vittoria Barzo 622x55 que aseguran un excelente agarre y durabilidad.

La horquilla Rockshox Judy Silver TK R de 100 mm proporciona una amortiguación eficaz para una conducción cómoda y controlada en terrenos difíciles. Además, los frenos de disco SRAM Level T DB garantizan una frenada fiable y segura en cualquier situación.

Complementan el conjunto el manillar 4ZA St

## Primera iteración: El texto es demasiado largo


In [123]:
prompt = f"""
Your task is to help a marketing team create a
description for a retail website of a product based
on a technical fact sheet.

Write a product description based on the information
provided in the technical specifications delimited by
triple backticks.

Use at most 50 words.

Technical specifications: ```{product_description}```
"""
response = get_completion(prompt)
print(response)

Disfruta de una bicicleta ligera y resistente con cuadro de aluminio 6061, transmisión SRAM NX Eagle de 12 velocidades y suspensión Rockshox Judy Silver TK de 100 mm. Ruedas Forza 29" y frenos SRAM Level T garantizan rendimiento y control en todo terreno. Garantía de 3 años.


## Segunda iteración; ¡El texto es demasiado general, la audiencia es Instagram!


In [ ]:
prompt = f"""
Your task is to help a marketing team create a
description for a retail website of a product based
on a technical fact sheet.

Write a product description based on the information
provided in the technical specifications delimited by
triple backticks.

The description is intended for an Instagram post.
Use Emojis.
We need the text in English
Use at most 50 words.

Technical specifications: ```{product_description}```
"""
response = get_completion(prompt)
print(response)

## Tercera iteración; Crea un JSON de resultado


In [ ]:
prompt = f"""
Your task is to help a marketing team create a
description for a retail website of a product based
on a technical fact sheet.

Write a product description based on the information
provided in the technical specifications delimited by
triple backticks.

The description is intended for an Instagram post.
Use Emojis.
We need the text in Euskera
Use at most 50 words.

The response MUST be a JSON with this format:
- name: Product name
- description: Instagram description in language Euskera
- category: Product category. The possible categories are: sports, swimming, mountain, cycling, running, rugby, combat
- details: An array with product details
- slug: Create a SEO optimized slug using the product name

Technical specifications: ```{product_description}```
"""
response = get_completion(prompt)
print(response)

#

<a id="lab3"></a>
## LAB 3 · Resumir

## Texto a resumir

In [16]:
blog_product = """
What is Life Pro Iso Micellar Zero?
Isolate Micellar Zero is a revolutionary and innovative protein supplement that allows you to enjoy the benefits of casein and protein isolate together.

It is made with the patented Lacprodan Micellpure Instant Sunflower protein. Isolate Micellar Zero is a protein of high purity and quality, which has not been mixed, and maintains all its properties.

Isolate Micellar Zero has a high purity of 80% to 94% depending on the chosen flavor. It has been designed in Life Pro's top flavors, including Choco Monky, Cocochello and Pinky Bun.

 Isolate Micellar Zero is derived from Danish milk; it is of high biological value and provides the 9 essential amino acids that the body cannot synthesize.

Isolate Micellar Zero is ideal for the preparation of recipes, since its texture is similar to that of conventional caseins. It can be taken at any time of the day: at breakfast, after training, throughout the day or at night before going to sleep.

Raw material: Lacprodan Micelpure Instant Sunflower
Isolate Micellar Zero is formulated with micellar casein isolate. This type of protein, revolutionary and innovative in the market, is characterized and differentiated by having the benefits of protein isolate, such as its purity and quality, together with the best of casein.

The patented raw material used is Arla's Lacprodan Micelpure Instant Sunflower. It has been produced with 100% native milk of Danish origin.

To obtain it, soft membrane filtration technology has been used. As a result, the protein is kept in its purest form and minimal denaturation occurs, maintaining the nutritional benefits.

Isolate Micellar Zero is between 80% and 94% pure, depending on the flavor chosen. Protein is the most important macronutrient, necessary for muscle mass gain and maintenance, as well as for the recovery process.

Digezyme
Life Pro Isolate Micellar Zero has been enriched with the patented digestive enzyme complex called Digezyme.

It contains the enzymes: alpha amylase, lactase, cellulase, protease and lipase. Digezyme facilitates the digestion and breakdown of fats, proteins and carbohydrates. It improves the processes of digestion and absorption of nutrients. In addition, it facilitates the intake for people with reduced enzyme levels.

Benefits and properties of Life Pro Iso Micellar
Patented raw material: Lacprodan Micelpure Instant Sunflower.
High purity: 80% - 94% protein depending on the flavor chosen. Not mixed with whey protein.
Enriched with the patented Digezyme enzyme complex.
It has the benefits and properties of isolate and casein.
High glutamine content.
Free of soy lecithin.
Contributes to the gain and maintenance of muscle mass. Promotes the recovery of muscle fibers damaged during training.
Ideal for the preparation of recipes thanks to its texture.
Instructions for use
To prepare Isolate Micellar Zero as a shake:

Dissolve 30 g (1 scoop) in water or milk (200 ml - 250 ml). Take 1-2 times a day, preferably at breakfast and after training.

To obtain a mousse type preparation:

Dissolve 30 g in water or milk 50-100 ml and mix until a thick and homogeneous texture is obtained.
"""

## Resumir el post y obtener la descripción corta del producto


In [17]:
prompt = f"""
Your task is to generate a short summary of a product \
review from an ecommerce site.

Summarize the review below, delimited by triple
backticks, in at most 30 words.

Review: ```{blog_product}```
"""

response = get_completion(prompt)
print(response)

Life Pro Iso Micellar Zero is a high-purity protein supplement combining micellar casein and isolate, enriched with digestive enzymes, ideal for muscle recovery, versatile in recipes, and free from soy lecithin.


## Resumir el post pero centrándose en algunos aspectos


In [129]:
prompt = f"""
Your task is to generate a short summary of a product \
review from an ecommerce site.

Summarize the review below, delimited by triple
backticks, in at most 30 words.

You must mention how to prepare the product.

Review: ```{blog_product}```
"""

response = get_completion(prompt)
print(response)

Life Pro Iso Micellar Zero is a high-purity protein supplement ideal for shakes or mousse. Prepare by dissolving 30g in 200-250ml water/milk for shakes or 50-100ml for mousse.


## Diferencia entre "resumir" y "extraer"


In [130]:
prompt = f"""
Your task is to extract relevant and important information of a product \
review from an ecommerce site. in at most 30 words.

You must mention the product composition.

Review: ```{blog_product}```
"""

response = get_completion(prompt)
print(response)

Life Pro Iso Micellar Zero is a high-purity (80%-94%) protein supplement made from patented Lacprodan Micelpure Instant Sunflower micellar casein isolate, enriched with Digezyme enzymes for improved digestion and muscle recovery.


## Generate multiple product reviews

In [20]:
review_1 = blog_product

review_2 = """
What is Tasty Rice Life Pro?
Pregelatinization is a process in which rice grains undergo a specific cooking process to change their traditional starch granule structure.

The texture of TastyRice makes it an instant source that can be enjoyed in just a few minutes. The result is Tasty Rice. A new way of providing carbohydrates from rice with a higher rate of absorption and hydration.

It is an ideal option to consume before training, and ensure that we have loaded glycogen stores, and ensure proper performance, as well as after for optimal recovery.

Its dissolution in milk, juice or water is much easier. It can also be used as a thickener to give textures to desserts, cakes, and many other recipes. Its neutral flavor allows it to be combined with all kinds of ingredients or flavorings.

Its Neutral Flavor
Tasty Rice flour has been elaborated with a neutral flavor to make it an ideal choice when combining it with other foods, adding your favorite flavorings, using it in recipes or with other sports supplements.

The benefits of pregelatinized flour
Pregelatinized rice is a source of carbohydrates of high glycemic index as well as a source of vitamins: B1, B2, B3 and B6, and minerals: Calcium, Phosphorus, Zinc.
The digestion process required is much faster than other conventional rice flours or carbohydrate sources.
It is an ideal alternative to avoid gastrointestinal discomfort since the pregelatinization facilitates the digestion process.
It is an ideal source of energy both before and after training to ensure recovery and loading of glycogen stores.
It is suitable for vegan and vegetarian diets.  It is GMO free.
Directions for use
Tasty Rice pregelatinized rice flour can be used as an ingredient in recipes, combined with other foods such as protein, nuts or incorporated into recipes as a thickener thanks to its creamy texture.

For its preparation, it is only necessary to mix the desired amount or a 50-gram dispenser with water, milk, vegetable drink or juice of choice until a thicker or thinner texture is achieved, depending on personal preference.
"""

review_3 = """
Composition of Life Pro Creatine Creapure 250g. neutral
This product is composed of creatine with a purity of 99.9%.

What is it for?
Life Pro Creatine Creapure comes to establish itself as one of the best options on the market when it comes to supplementing your workouts. It contains the best creatine formula, called Creapure, which reaches an amazing 99.9% purity.

This feature is of vital importance, as it means that the properties of creatine can be enjoyed to the fullest with Life Pro Creatine Creapure 250g. It could be said that this supplement improves the performance of any athlete at a generic level, but let's try to detail its most outstanding benefits.

Creatine facilitates the synthesis and regeneration of phosphocreatine, one of the most important fuels for sports activity, since it is responsible for anaerobic efforts, which are the most explosive but also those that are exhausted sooner. In this way, Life Pro Creatine Creapure has the ability to improve attributes such as strength or power, ideal for you to perform that extra repetition that, in the medium-long term, will be the key to gain lean mass and take your performance a step further.

Life Pro Creatine Creapure also promotes protein synthesis, so it is very useful for maintaining optimal muscle health. The same is true for the increase of glycogen present in the muscles, a feature that enhances performance at a generic level very beneficial for any sporting discipline.

In short, Life Pro Creatine Creapure creatine can provide its properties in many different contexts, but it always does so to improve your physical attributes.

Benefits of Life Pro Creatine Creatine Creapure 250g. neutral
Creatine with a purity of 99.9%.
Facilitates the regeneration of phosphocreatine.
Increases strength and power.
Helps in the growth of muscle mass.
Promotes protein synthesis.
Increases muscle glycogen.
Improves overall athletic performance.
How to take Life Pro Creatine Creapure 250g. neutral?
Take 3 grams before training. You can repeat the intake after training.

How to use
The basic recommended dose is 3 grams (1 scoop) or 0.1g per kg of body weight, dissolved in 250 ml of water, before training. Days of NO training: You can rest or continue with the same dosage.
"""

review_4 = """
What is Life Pro Rage Pro?
Life Pro Rage is a sports supplement to be taken before training to boost physical capacities and mental focus in the most optimal way.

Life Pro's new pre-workout has been developed with a selection of carefully selected ingredients, in order to offer you a pre-workout with an innovative formulation, which adapts to your workouts and helps you to get the maximum potential out of each one.

The main objective of Rage Pro is to increase energy and pump or vasodilation. Each serving of Life Pro Rage provides 300mg of caffeine and 8 grams of citrulline malate in a 2:1 ratio and 4 grams of dry amaranth extract.

All the ingredients are of the highest quality, and have been patented, as in the case of Oxystorm Dried Amaranth Extract. It has been sweetened with sucralose.

Life Pro Rage has been designed in 2 different flavours: energy and lollipop.

What are the components of Life Pro Rage?
Life Pro Rage has been formulated with carefully selected, patented and high-quality ingredients in order to provide you with a highly effective pre-workout.

 L-citrulline-DL-malate:

Citrulline malate results from the combination of two elements: a non-essential amino acid called citrulline with the malate salt. Citrulline malate is converted in the body into L-arginine. It is a precursor of this substance and nitric oxide.

As a result, nitric oxide and arginine promote muscle congestion, increase vasodilation and the arrival of nutrients and oxygen to the muscle tissues more quickly. Other functions include promoting cardiovascular health, improving the immune system, reducing the feeling of fatigue and promoting energy production at the cellular level.

Each serving provides 8 grams of Citrulline Malate.

Oxystorm Amaranth Dry Extract:

The dry extract of amaranth used in the formulation of Life Pro Rage has the Oxystorm patent. Amaranth is considered a superfood due to its numerous properties and benefits for the body.

It promotes vasodilation, and as a result, promotes better blood flow to the muscles, as well as helping to increase performance and prevent tiredness or the onset of fatigue.

Each serving provides 4 grams of dry amaranth extract.

Caffeine anhydrous:

Caffeine anhydrous is a type of quick-release caffeine. Its effects are felt within 15 to 30 minutes of taking it.

Caffeine is one of the star supplements in sports. It is an ally for getting the most out of every workout, and achieving a rapid increase in performance and strength. Among other benefits, it helps to postpone the onset of fatigue and tiredness. Caffeine also has a thermogenic effect. It contributes to weight and body fat loss and increases body temperature.

Each dose provides 300 mg of caffeine.

The flavours of Life Pro Rage.
Life Pro Rage has been sweetened in its 2 flavours with sucralose. It does not contain aspartame or other sweeteners. Sucralose is a sweetener suitable for diabetics because it does not affect blood sugar levels.

It has the capacity to sweeten 500 to 600 times more than conventional sugar. It is obtained from sucrose.

Life Pro Rage has been designed in two different flavours:

-Energy: an intense energy drink flavour.

-Lollipop: a delicious sweet, cherry, lollipop-like flavour.

Sucralose contains no calories: it provides sweetness and flavour only, being a very safe sweetener for consumption. It does not alter the feeling of openness and/or satiety.

What are the benefits of Life Pro Rage?
Its combination of patented ingredients is one of the most effective for improving performance and getting the most out of every workout. Improves strength and endurance levels.
Helps increase energy levels and postpone the onset of fatigue and tiredness.
Improves the total volume of training supported.
Its components promote pumping and vasodilation. As a result, nutrients are optimally delivered to muscle tissue and post-workout visual sensation is improved.
Caffeine increases mental clarity, focus and alertness. It is rapidly absorbed. Caffeine has a thermogenic effect: it can support fat loss.
When to take Life Pro Rage Pro?
It is recommended to take Life Pro Rage, being a pre-workout with caffeine, within 20 minutes to 1 hour before training, and trying to avoid taking it close to sleep (4 to 6 hours).

Life Pro Wild Pump provides 300 mg of caffeine. Each person has a different sensitivity to caffeine. Each person's dose may vary. Avoid consumption close to sleep.

How to take Life Pro Rage?
Dissolve 18 g (2 scoops up to the 20 ml mark) in water or juice (200 ml - 250 ml). Take 40 minutes before training.

Consuming a caffeinated pre-workout close to sleep may interfere with the quality of sleep and rest.
"""

reviews = [review_1, review_2, review_3, review_4]

In [ ]:
for i in range(len(reviews)):
    prompt = f"""
    Your task is to generate a short summary of a product \
    review from an ecommerce site.

    Summarize the review below, delimited by triple \
    backticks in at most 50 words.

    Act like a sarcastic nutritionist who explains everything \
    with football metaphors (signings, goals, line-ups, referees).

    Always in spanish

    Review: ```{reviews[i]}```
    """

    response = get_completion(prompt)
    print(i, response, "\n")

In [ ]:
for i in range(len(reviews)):
    prompt = f"""
    Your task is to classify a product \
    review from an ecommerce site.

    Classify the review below, delimited by triple \
    backticks in at most 20 words.

    Categories are: Protein, Creatine, Food, Pre-workout, Other

    Only return the category, nothing else

    Review: ```{reviews[i]}```
    """

    response = get_completion(prompt, model="gpt-4.1-nano")
    print(i, response, "\n")

<a id="lab4"></a>
## LAB 4 · Clasificar

## Curriculum Vitae

In [23]:
cv = """

Nombre y Apellidos: Jordi Miró
Dirección: Calle de la Innovación, 123, 08018 Barcelona
Teléfono: 612 345 678
Correo Electrónico: jordi.miro@ejemplo.com
LinkedIn: linkedin.com/in/jordi-miro

Objetivo Profesional
Desarrollador de software con más de 12 años de experiencia, especializado en el desarrollo de aplicaciones web y móviles. Busco un rol desafiante en una empresa tecnológica innovadora donde pueda aplicar mis habilidades en codificación, resolución de problemas y gestión de proyectos para impulsar la eficiencia y la innovación.

Experiencia Profesional

Desarrollador de Software Senior
Soluciones Innovatech, Barcelona
Junio 2018 – Presente

Dirigí un equipo de desarrolladores en la creación de una plataforma de comercio electrónico que ha aumentado las ventas en un 150% en el primer año.
Implementé metodologías Ágiles y Scrum, mejorando el tiempo de entrega del proyecto en un 40%.
Desarrollé y mantuve más de 20 APIs RESTful para integraciones de terceros, mejorando la interoperabilidad del sistema.

Desarrollador de Software
TechAdvance, Madrid
Abril 2014 – Mayo 2018

Contribuí en el desarrollo de una aplicación móvil que ganó el premio a la Mejor Aplicación del Año en 2016.
Optimicé el rendimiento de aplicaciones existentes, logrando una mejora del 25% en la velocidad de carga.
Colaboré en equipos multidisciplinarios para integrar nuevas funcionalidades, siguiendo las mejores prácticas de seguridad y código.

Desarrollador Junior
Innovaciones StartApp, Valencia
Septiembre 2010 – Marzo 2014

Desarrollé funcionalidades para aplicaciones web utilizando HTML, CSS, JavaScript y frameworks como React y Angular.
Participé en la planificación de proyectos y revisión de código, asegurándome de que los estándares de calidad se cumplieran.
Implementé soluciones de software personalizadas para clientes locales, mejorando su eficiencia operativa.

Educación
Máster en Ingeniería del Software
Universidad Politécnica de Cataluña, Barcelona
2010 - 2012

Grado en Informática
Universidad de Valencia, Valencia
2006 - 2010

Competencias Técnicas
Lenguajes de Programación: Java, Python, JavaScript, PHP
Frameworks y Bibliotecas: React, Angular, Node.js, Django
Bases de Datos: MySQL, MongoDB, PostgreSQL
Herramientas y Plataformas: Docker, Git, Jenkins, AWS
Metodologías Ágiles: Scrum, Kanban

Idiomas
Catalán: Nativo
Español: Nativo
Inglés: Avanzado

Intereses Profesionales
Apasionado por la tecnología emergente, el desarrollo de software sostenible y la creación de soluciones innovadoras que resuelvan problemas reales. Miembro activo de la comunidad de código abierto, contribuyendo regularmente a proyectos y asistiendo a hackathons.

"""

## Categorización general

In [24]:
prompt = f"""
You are an expert in categorizing resumes (CVs) into general job categories
based on candidates' work experience, technical skills, and academic background.

We only need the category don't explain how you arrived to the conclusion

Categories should be:
- Software developer
- Telecomunications engineer
- Administrative
- Technical

CV: ```{cv}```
"""

response = get_completion(prompt)
print(response)

Software developer


## Extracción de competencias


In [25]:
prompt = f"""

You are an expert in meticulously examining resumes (CVs) to extract and list
the candidates' competencies. Your task involves categorizing these competencies
into technical skills and languages.
Your goal is to provide a detailed inventory of skills
and qualifications for each candidate, ensuring a clear understanding of their
professional capabilities.

We only need the categories don't explain how you arrived to the conclusion

We want the response in a JSON

CV: ```{cv}```
"""

response = get_completion(prompt)
print(response)

```json
{
  "technical_skills": {
    "programming_languages": [
      "Java",
      "Python",
      "JavaScript",
      "PHP"
    ],
    "frameworks_and_libraries": [
      "React",
      "Angular",
      "Node.js",
      "Django"
    ],
    "databases": [
      "MySQL",
      "MongoDB",
      "PostgreSQL"
    ],
    "tools_and_platforms": [
      "Docker",
      "Git",
      "Jenkins",
      "AWS"
    ],
    "methodologies": [
      "Scrum",
      "Kanban"
    ],
    "other_skills": [
      "API RESTful development",
      "Agile methodologies",
      "Project management",
      "Code review",
      "Performance optimization",
      "Security best practices"
    ]
  },
  "languages": {
    "Catalan": "Native",
    "Spanish": "Native",
    "English": "Advanced"
  }
}
```


## Extracción de información relevante para guardar en una BBDD


In [ ]:
prompt = f"""

You are an expert in comprehensively analyzing resumes (CVs) to extract all
pertinent information, organizing it into a structured JSON format. Your
task is to focus on key areas such as the candidate's name, years of
experience, general job category, technical competencies, language skills,
educational background, and a brief description of the candidate in no more
than 50 words. For any fields where information is not available, you should
explicitly mark them as null. Your goal is to provide a detailed, organized,
and easily accessible overview of each candidate's professional profile,
ensuring that the JSON object you create is both accurate and informative.

Today is: 03/10/2026

Example of JSON Structure:

candidate_name: Candidate Name,
years_of_experience: 10,
job_category: Software Development,
technical_skills: [Python, JavaScript, React],
language_skills: [English, Spanish],
education: [B.Sc. Computer Science, M.Sc. Software Engineering],
brief_description: An experienced software developer with a strong background in web and mobile application development. Skilled in leading teams and innovative project delivery.

Instructions:

Candidate Name: Extract the full name of the candidate.
Years of Experience: Calculate the total years of professional experience.
Job Category: Determine the general category of employment based on the resume.
Technical Skills: List all technical skills mentioned, such as programming languages, software, and tools.
Language Skills: List all language proficiencies.
Education: Provide a list of educational achievements, including degrees and certifications.
Brief Description: Summarize the candidate's professional profile in up to 50 words. This should highlight their key achievements, strengths, or specialties. If any information is missing, ensure to mark those fields as null to maintain data integrity.
Your meticulous approach in transforming these details into a structured JSON format will greatly facilitate the evaluation and comparison of candidates' profiles for potential employers or recruiters.

CV: ```{cv}```
"""

cv_json = get_completion(prompt, model=DEFAULT_MODEL)
print(cv_json)

## Comparación

In [27]:
offer = f"""
Requisitos
Estudios mínimos
Educación Secundaria Obligatoria
Experiencia mínima
No Requerida
Conocimientos necesarios
HTML
CSS
JavaScript
Git
Github
Requisitos mínimos
¡La Velada 4 ya está aquí! ¿Quieres formar parte de la creación de su web?

Si eres programador/a, este proyecto te va a encantar porque podrás crear la web de La Velada 4 conmigo de la mano de InfoJobs.

¿Estás preparado/a para programar una web que lo cambie todo y ser parte de la historia?

¡Inscríbete ya!

Descripción
Busco 2 programadores/as que tengan tantas ganas y habilidad para mover las manitas.

Y además:

-Capacidad para solucionar problemas de forma creativa y eficiente.

-Creatividad para crear la nueva web que haga explotar la cabeza de lo guay

-Ganas de trabajar en equipo para hacer algo espectacular

Conocimientos valorables:
cualquier framework de Frontend (React, Angular, Vue, Astro, Next.js...)

Ofrecemos:

- 1.200€ netos por dos semanas de trabajo

¡Inscríbete ya a la oferta de trabajo!

Midudev e InfoJobs gestionarán el proceso de selección y cederán los datos de las personas seleccionadas a una ETT con el fin de formalizar el contrato de trabajo.

Tipo de industria de la oferta
Programación, Consultoria y otras Activ. informaticas
Categoría
Informática y telecomunicaciones - Programación
Nivel
Empleado/a
Número de vacantes
2
Salario
Salario: 1.200€ - 1.300€ Bruto/mes
Beneficios sociales
Flexibilidad horaria
Teletrabajo
"""

In [ ]:
prompt = f"""

You are an expert in comparing job offers with job candidates.

You must analyse the [Job candidate] and the [Job Offer] and you have to look
for the common points between the job offer and the candidate.
With what you find, generate a friendly email to be able to respond
to the candidate automatically telling them that we will analyze their
candidacy if the CV is similar to the job offer. Otherwise,
inform you that your CV is very interesting and that we save your
information in our Database

The email must be in spanish

[Job candidate]: ```{cv_json}```
[Job Offer]: ```{offer}```
"""

response = get_completion(prompt)
print(response)

<a id="lab5"></a>
## LAB 5 · Inferir

## Sentiment analysis

In [29]:
messages = [
    "¡Increíble partida! No podía dejar de mirar, ¡qué final!",
    "Este streamer no sabe lo que hace, muy aburrido...",
    "¿Alguien más está teniendo problemas con el stream? Se corta a cada rato.",
    "Me encanta este canal. Siempre buen contenido y buena onda :)",
    "Que esta haciendo ahora este soplapollas?"
]

In [135]:
for i in range(len(messages)):
    prompt = f"""

    You are an expert in conducting sentiment analysis on user-generated content.
    Your current task involves analyzing a set of comments from Twitch to determine
    the underlying sentiment: positive, negative, or neutral. For each comment,
    you will assess the tone, language, and context to classify its sentiment
    accurately.

    This analysis will help in understanding viewer engagement,
    content reception, and potential areas for improvement or commendation
    for streamers.

    You MUST only provide the result of "positive", "negative" or "neutral"

    message: ```{messages[i]}```
    """

    response = get_completion(prompt, model="gpt-4.1-mini")
    print(i, response, "\n")

0 neutral 

1 neutral 

2 neutral 

3 neutral 

4 neutral 

5 neutral 

6 negative 

7 neutral 



## Extraer información relevante

In [31]:
review = """
El problema del material de estas calleras de PICSIL es que no tiene grip para hacer
todos los ejercicios de gymnastics. Para halterofilia puede servir pero para
gymnastics recomiendo coger otras que sean de otro tipo de piel que no tenga
ese acbaado. A mi me encanta la marca de PICSIL, pero no este diseño.
Me compré las calleras para substituir las viejas y sigo utilizando las
viejas porque con las nuevas me resbalo.
"""

In [32]:
prompt = f"""
Identifica los siguientes elementos del texto de la reseña:
- Artículo comprado por el reseñador
- Compañía que ha fabricado el artículo

La reseña está delimitada con comillas triples invertidas. \
Formatea tu respuesta como un objeto JSON con \
"artículo" y "marca" como claves. \
Si la información no está presente, utiliza "desconocido" \
como valor.
Haz tu respuesta tan corta como sea posible.

Texto de la reseña: '''{review}'''
"""
response = get_completion(prompt)
print(response)

```json
{"artículo":"calleras","marca":"PICSIL"}
```


## Más de una tarea a la vez


In [ ]:
prompt = f"""
Identifica los siguientes elementos del texto de la reseña:
- Sentimiento (puede ser solo positivo o negativo)
- ¿El reseñador expresa enojo? (verdadero o falso)
- Artículo comprado por el reseñador
- Compañía que ha fabricado el artículo

La reseña está delimitada con comillas triples invertidas. \
Formatea tu respuesta como un objeto JSON con \
"is_positive", "anger", "article" y "brand" como claves. \

is_positive es un booleano: true si el sentimiento es positivo, false si es negativo.
Si la información no está presente, utiliza "desconocido" \
como valor.
Haz tu respuesta lo más corta posible.

Texto de la reseña: '''{review}'''
"""
response = get_completion(prompt, model="gpt-4.1-mini")
print(response)


## Extraer categorías

In [34]:
transcript = """
Ayer fue un día histórico Y es que no tuvimos solo una de esas noticias que se puede considerar noticia del año sino que tuvimos dos y es que Google presentó gemini 1.5 un modelo multimodal con una ventana de contexto enorme y unas capacidades sorprendentes pero esta noticia quedó en nada después del anuncio de Open Ai Y es que Open a presenta sora un modelo de generación de vídeo que no tiene nada que ver con los modelos actuales directamente fulmina la competencia creando vídeos con una consistencia temporal increíble una calidad alucinante pero que además va mucho más allá de ser un mero generador de vídeo tal y como plantea Open a puede ser la puerta a modelos de simulación de mundos es algo completamente alucinante es un modelo que puede cambiar al paradigma y acercarnos mucho más a la ig que las últimas innovaciones que hemos visto en modelos de generación de lenguaje y hoy me voy a centrar solo en esta noticia y habrá tiempo tiempo para gemini pero hoy la noticia es sora dentro vídeo y vamos a empezar directamente por lo que es la gran noticia del día de ayer el nuevo modelo de generación de vídeo de Open sora y lo que estás viendo ahora mismo este fondo de la página web es un vídeo generado con esta tecnología pero es que no han mostrado un vídeo han mostrado una cantidad ingente de ellos y lo que se puede apreciar los detalles que se pueden apreciar en Ellos nos hace pensar que estamos ante algo muy especial Algo totalmente diferente a lo que hemos visto hasta ahora lo que os voy a enseñar hoy no tiene nada que ver ni con Rango ni con pica por muchos motivos ya no solo por la calidad de imagen ni tan siquiera por el hecho de permitir vídeos muchos más largos veremos que hay detalles que van incluso Más allá de esto pero lo mejor es ver primero algunos ejemplos de lo que puede hacer sora para que simplemente alucinosis artificial no esperaba que llegásemos a estos resultados Ni tan pronto ni a este nivel yo me imaginaba que quizá podríamos llegar a un nivel similar pero con clips más cortos de alrededor de 10 segundos y de Cara a finales de año lo de lo que estamos viendo ahora mismo es una auténtica locura de modo que vamos a ver algunos ejemplos de lo que puede hacer sora porque en la página web hay varios Y es que todos son absolutamente espectaculares Mirad Este vídeo Cuántos elementos aparecen en escena mirar la coherencia temporal hay defectillos si nos fijamos constantemente en las piernas de la chica vemos que hay algún movimiento o algún artefacto extraño pero aú así es absolutamente alucinante los reflejos el resto de personas la consistencia de los símbolos todo es simplemente increíble no hay nada parecido a esto a día de hoy y si avanzamos un poco más podemos ver Incluso el detalle el detalle de la piel esto es vídeo que si lo ves muy rápido se te puede colar perfectamente como vídeo real pero es que no han mostrado un único ejemplo en esta primera serie hay un total de nueve ejemplos que podemos ver y fijaros por ejemplo en esto este sabemos que no es real porque los mamuts pues no existen se extinguieron pero el nivel de detalle de lo que estamos viendo Es nivel de detalle de muy buen cgi tal y como yo lo estoy viendo Es decir en un primer vistazo es calidad completamente fotorrealista y en este aspecto es muy interesante abordar este aspecto la comparación con el cgi porque os voy a enseñar ejemplos de vídeos que artistas de cgi han dicho esto es muy difícil de hacer con cgi y este modelo de Inteligencia artificial es capaz de hacerlo solo lo vamos a ver un poco más adelante vamos a ver un poco todos ejemplos para que veáis lo que puede hacer Y fijaros es que movimientos de cámara consistencia total esto es Es que es absolutamente increíble es la la palabra es que quizá mira que cada día estoy viendo novedades de Inteligencia artificial y esto simplemente me ha dejado completamente descolocado de hecho estuve haciendo dos entrevistas para abordar este tema porque el tema de la Revolución visual ya os digo creía sinceramente que iba a a ser el gran tema de este año y hablé hice una entrevista a una directora de cine que lleva años experimentando con Inteligencia artificial con Jos que ya lo traje en un directo hace un tiempo en el canal hablando de precisamente Esto del gran problema de la consistencia temporal en la generación de texto a vídeo y de repente termino la entrevista abro internet y me encuentro con esto es que es es una locura de verdad hay veces que la vida te sorprende y yo ya creía que con la Inteligencia artificial me iba a sorprender pero en pequeñas dosis pero no que iba a contemplar un salto de calidad tan acusado tan bestia por decirlo rápido y mal en en uno de los ámbitos y este es que es increíble fijaros aquí y en esto vamos a indagar después en la física de las de las olas porque luego os enseñaré un un comentario que ha hecho Shin fan de de envidia diciendo lo sorprendente que es esto la capacidad para recrear unas olas que a simple viste parecen tan realistas la dinámica de fluidos es otro de estos temas que es super complejo de representar en cgi y aquí estamos viendo un vídeo que repito seguro que si nos fijamos bien no es perfecto pero es totalmente aparente es totalmente funcional Es simplemente increíble y aquí hasta ahora hemos visto vídeos de cosas reales de cosas que pueden estar en el set de entrenamiento pero es que fijaros en este otro vídeo aquí está ya representando un animal o un personaje estilo Pixar y Bueno hay defectos en el fondo por ejemplo este fondo tiene un parx que no debería tener y hay detalles pero es que simplemente sorprendente si te están diciendo que esto es un fragmento de una película de Pixar de entrada yo creo que cuela que te lo crees después te fijas en los detalles y verás no hay hay cosas que que no funcionan pero estamos ahí estamos en el punto donde en cuanto tengamos más control porque esto es la otra cosa que que tengo ganas de comprobar si esta herramienta nos da un control muy preciso sobre lo que podemos generar en las imágenes que es es el otro gran caballo de batalla con el vídeo con Inteligencia artificial si podemos tener un control preciso sobre el vídeo que vamos a generar la calidad que estamos viendo y la duración de los clips es más que suficiente para articular cualquier producción audiovisual es muy fuerte Yo Nunca creí que diría algo así a estas alturas estamos a un paso de poder crear cine con Inteligencia artificial la calidad está ahí la consistencia temporal está ahí la duración de los clips está ahí la la única duda que crea para despejar en esta ecuación es si podemos tener un control suficientemente preciso y estoy seguro que sí Porque para mí de todos los elementos de la ecuación Este es el que me parece Aparentemente al menos más fácil de solucionar por lo que estamos viendo literalmente el futuro como nos da una bofetada en la cara y esto a mí es incluso lo voy a decir creo que nos pone más cerca de una posible Inteligencia artificial general lo que estamos viendo ahora que el siguiente salto que veremos en los modelos de generación de lenguaje porque hay mucho más y luego lo veremos en el en la noticia compartido el propio Open a de cómo han creado Este modelo hay mucho más matices y mucho más más allá del simple hecho de encadenar frames que sean coherentes hay como un conocimiento del mundo de cómo funcionan las cosas para poder generar esto y que sea casi casi capaz de quebrantar El Valle inquietante y luego lo veremos con la explicación que ha hecho Open Ey de cómo ha creado Este modelo y también con algunas reflexiones por ejemplo como la de gin fan Pero bueno podríamos estar viendo vídeos constantemente Mirad este este es otro detalle luego lo comentaremos pero es capaz de crear vídeos en formato horizontal en formato vertical y vídeos como este este es el vídeo que yo realmente os quería Mostrar fijaros la dinámica de fluidos que estamos viendo aquí fijaros Cómo abraza el barco este no no sí que flota y le vemos artefactos extraños pero este de la izquierda Realmente está muy muy logrado y pensemos que esto es la primera versión del modelo que estamos viendo esto evolucionará porque Este es otro de los puntos sorprendentes de que cuando me he estado informando Open ahí dice que del modo que han entrenado estos Este modelo para generar estos vídeos es fácilmente escalable ven como una progresión en la relación entre el tiempo de entrenamiento y los resultados que es como que tiene una correlación es decir que si le meten más capacidad de cálculo al sistema y optimizan un poco pueden conseguir modelos superiores Pero bueno lo que estamos viendo aquí ya te deja completamente boqu abierto como os decía el otro día estaba entrevistando precisamente una directora de cine que lleva experimentando con Inteligencia artificial desde hace 6 años y que precisamente organiza la segunda edición de un festival de cine aquí en Barcelona el el rfm festival y me lo dijo bueno el año pasado lo lanzamos antes de que saliese runway y se presentaron cochas muy muy interesantes pero el Boom fue cuando apareció runway Pues ahora tienen abierta la segunda convocatoria con runway y van seguro que van a recibir muchos más vídeos pero sin que esto esté aún disponible por lo que para la tercera edición yo creo que ya van a recibir películas o cort metrajes de una calidad similar a lo que podemos crear con cámaras normales y convencionales es es una locura de hecho es que estoy recordando la conversación que la tuve antes de ayer con con Ana y y simplemente la sensación es que se nos ha quedado en algunos aspectos obsoleto alguno de los comentarios porque esto ni ella ni yo nos imaginamos que estuviese tan tan cerca decir también que el punto negativo Es que esto aún no está disponible para todo el mundo de momento está en manos de un equipo que está comprobando que no pueda generar material peligroso y a manos de algunos artistas diseñadores y creadores de vídeo creadores audiovisuales que les puedan dar feedback de modo que no sabemos cuando esto estará disponible por ejemplo dentro de chat gpt estás Es que os imagináis en chat gpt poder decirle créame un vídeo y que no pase como ahora con envidio que te hace cuatro refritos de imágenes de stock sino que te genere un vídeo entero con los cortes la música La la locución es que estamos ahí estamos ahí de un día a otro estamos ahí esto es Yo creo que es tan revelador como en el momento en que se descubrieron la arquitectura de Transformers y vimos que por primera vez se podía articular los lenguajes y vios su potencial pero esta además es más visual es más impactante fijaros también en otro detalle que he visto y es en la complejidad de los vídeos que puede llegar a generar Cuántos elementos que tienen su propio movimiento su propia escala hay en este vídeo observad el movimiento increíblemente complejo que realiza la cámara y cómo todo aguanta bastante coherente lo vuelvo a decir si lo miramos con lupa seguro que hay defectos aquí desaparece un personaje pero es que da igual es que dónde estábamos antes de ayer y dónde estamos ahora Esto es lo que debemos mirar y dónde estaremos dentro de yo ya digo se meses porque esto a partir de aquí sí que van a evolucionar las cosas es decir ahora ya tenemos otro Punto de partida desde donde volver a propulsar noos para saltar más más hacia delante fijaros en estos reflejos cómo simula los reflejos que hay en el exterior mientras anima al personaje que está dentro es una locura esta simulación física quedaos con esto porque es lo que voy quiero comentar en la segunda parte fijaros en en en este ojo Es que estamos mirando algo que a los humanos nos genera muy rápidamente el vaya inquietante la mirada es lo más difícil de conseguir replicar de forma artificial ya sea con Inteligencia artificial o sea con con cgi y no es perfecto pero ostras da mucho mucho el pego fijaros otro en estilo animación las texturas los personajes de fondo la iluminación la coherencia de cada uno de los elementos luego veremos Cómo lo han conseguido esto porque han utilizado una aproximación un poco distinta a a otras estrategias Y la verdad es que merece la pena también echarle un vistazo Este vídeo sí que tiene un defecto extraño pero en su defecto extraño es diría bastante interesante de hecho fijaros que en este cambio de de movimiento vemos como si ellos estuviesen en el mismo nivel pero luego vemos como si estuviese en una terraza Es una transición que para una película si estuviese justificada sería chulísima ahora para mí es un poco como un error en en Cómo se mueve la cámara que pero lo hace con una consistencia que incluso el fallo el error en que hace que este vídeo no sea creíble lo hace interesante es alucinante estoy viendo muchos vídeos estoy comentando muchos ejemplos porque creo que hoy merece la pena detenernos a esto porque como os he dicho al principio del vídeo yo creo que estamos viendo historia al menos de la de la tecnología historia de la ciencia esto es brutal es que bueno no voy a comentar mucho más porque más o menos la mayoría de detalles ya los he comentado Este vídeo fijaros en este vídeo porque también es muy interesante el hecho de que aquí tenemos multitud de vídeos que son esencialmente independientes y algunos muestran una coherencia muy muy elevada es otra cosa que hay que tener en cuenta un poco como lo otro como la la escena que hemos hemos visto del de la fiebre del Oro de de California en la época de la fiebre del Oro ahí había muchos elementos pero era la misma secuencia aquí estamos viendo muchos elementos pero son elementos que tienen cierta Independencia pero que a la vez deben integrarse en una misma atmósfera es complicadísimo hacer esto Yo estoy seguro que si me está viendo alguien que trabaja en efectos especiales en cgi sabe lo difícil que es integrar muchos elementos para que se vean coherentes y este modelo lo hace literalmente solo es que ya os Digo vamos a ver películas de animación a muy muy corto plazo es increíble aquí aquí debajo No los he ido comentando pero también lo podemos hacer es ver el tipo de proms que utilizan es como hacer un prom para Dali 3 es que realmente algunos son muy muy cortitos muy muy sencillos y se consiguen ya resultados espectaculares la incógnita es el control pero es que me parece algo tan relativamente sencillo de solucionar que vamos yo creo que el año que viene no sé si películas pero cortometrajes de una calidad casi indiscernible de como mínimo películas hechas con cgi vamos a ver porque esto es es muy loco no no no se puede decir de otra manera fijaros esto lo que nos dicen puede generar escenas complejas con múltiples personajes diferentes tipos de movimiento y mantener eh precisos los detalles entre el el personaje o el elemento principal y el fondo es lo que nos dicen es que el modelo entiende No solo lo que el usuario le ha dado en el en la indicación pero también y esta esta es la clave de todo todo aquello que existe en el mundo físico yo creo que podríamos seguir viendo ejemplos todos tienen algún detalle este es alucinante no no no no se puede decir de otro modo es es un vídeo de esos que tú lo ves y yo no soy capaz de decir que esto no es un vídeo real a simple vista la calidad del vídeo es enorme y ese momento momento en que pasa delante de de un muro o similar y cambia el reflejo es decir pasamos de ver que la chica se refleje levemente cuando hay mucha luz en el exterior a reflejarse completamente cuando se opaca esta luz Esto es algo es lo que os decía antes este es el vídeo El ejemplo perfecto de algo que es muy pero que muy complejo de recrear con cgi y esto lo hace un modelo a partir de la indicación de texto no sé ni Cuánto tarda el tiempo de inferencia no sé los recursos que consume lo único que sé es que esto es es una locura todo lo que estoy viendo para mí a ojo humano y a simple vista es metraje que podría ser real y alguno de estos metrajes es es cosas así son cosas completamente oníricas y fantasiosas se abre una nueva era a la hora de crear vídeo a hora de crear de la creación audiovisual pero yo creo que esto va más allá como os he dicho esto creo que nos deja más cerca de la II o de la creación de una Inteligencia artificial general que las últimas mejoras que habíamos visto con los modelos de lenguaje que a mí me habían dejado un poco tibio De hecho mi con los modelos de lenguaje Es que este año no íbamos a ver otro salto similar de gpt3 a gpt 4 Ahora yo lo pongo en duda después de ver lo que acaba de hacer Open Ah ahí seguramente igual patina un poco pero con la generación de vídeo yo creía que podíamos llegar a algo similar pero no a este nivel sinceramente y no en febrero yo Esto me lo imaginaba con suerte para diciembre noviembre finales de año que todos los modelos que están saliendo que permiten hacer cosas concretas relacionadas con el vídeo se entendiese Cómo poder introducirlos en en en un único modelo las diferentes estrategias y de ahí mejorar mucho pero no Open a lo ha conseguido y lo ha conseguido con una aproximación muy interesante y esto creo que es lo siguiente que tenemos que ver porque esto va más allá del vídeo y antes de ir a explicar cómo han entrenado Este modelo que lo hace diferente de otros modelos de generación de vídeo cómo se han aproximado él Creo que merece la pena comentar este post de jin fan para entender que esto realmente va mucho más allá de un simple generador de vídeo es decir él lo define como que sora es un motor de física dirigido por datos es decir es una SIM es capaz de simular el el muchos mundos de manera real y fantástica y lo bueno es que este simulador aprende a a renderizar de un modo muy sofisticado y intuitivo físicas y Horizontes de consistencia a largo plazo y u Este vídeo de muestra que os he enseñado antes el vídeo del café para explicarnos todas las locuras que pasan aquí y cómo cada una de estas locuras esencialmente es como un ámbito de la simulación de de fluidos de la representación gráfica Y cómo esto lo soluciona la Inteligencia artificial con un único modelo tú vas aquí y tú lo ves tienes como dos elementos de 3D que serían los barcos pirata con decoraciones diferentes y la el modelo ha sido capaz de solucionar esta recreación que en principio es una recreación completamente inventada dentro de su espacio latente los objetos 3D están animados consistentemente y la las velas evitan el el camino de uno y otro objeto es decir ha conseguido generar una coreografía consistente entre los dos barcos las dinámicas de fluido del café incluso de la espuma fijaros que este café tiene espuma es algo muy muy muy loco se forma que se forma al ror de los barcos y y nos dice que ostras que da mucho el pego y que la simulación de fluidos es literalmente un campo entero dentro de la de los gráficos por ordenador yo Bueno cuando estudié aeronáutica tuve un compañero que se dedicó a estudiar en profundi la dinámica de fluidos y era una locura lo que necesitaba a nivel de de de potencia de cálculo computacional para hacer sus simulaciones clo tú ves algo así dias igual solo es la recreación visual pero es que debajo debajo esta coherencia se tiene que entender de algún modo y es claro es es sorprendente otra cosa es el fotorrealismo es Casi casi como un renderer con retracing es decir un render a a la máxima potencia de esos que tardan horas de pasar un retracing por un por un render es decir simular los diferentes Rayos de Luz con una cantidad suficiente de rayos de luz y hacer el cálculo Esto es lo que aumenta muchísimo el el tiempo de renderizado de de un cgi pues estos modelos no sabemos el tiempo que tardan lo repito Me da igual con los resultados aunque tarde tanto más que un render todo lo que te ahorras por el otro lado creo que lo justifica pero es que estamos consiguiendo esto fotorrealismo e a un nivel que se necesita un renderista experto eh para conseguir la configuración adecuada para que de El pego Y que además Eh pues funciona de un modo seguramente más rápido en los tiempos de inferencia seguramente serán menores que que lo que tarda hacerse un render bueno tienen que ser menores porque yo creo que si lo han presentado es porque creen que lo pueden poner a disposición de un modo u otro de del consumidor a nivel comercial y evidentemente si esto tardas al mismo tiempo a renderizar que lo que tarda un render pues no sería no Sería posible fijamos bien como el ador tiene en cuenta pequeñas partes del del de la copa de café comparada con océanos y aplica conceptos como el til shift photography el til shift Son unos objetivos de fotografía que se pueden descentrar su eje y que generan esos vídeos o esas fotografías con efecto miniatura pues esto también se puede percibir Dentro de este vídeo para que esto tenga sentido y lo más loco es que lo que estamos viendo aquí es lo que os decía no es una escena que podamos ver en el mundo real es algo completamente imaginario y ha sido capaz de recrearlo por lo que el potencial de este modelo es increíble No no me voy a cansar de repetirlo en este vídeo sé que hoy igual peco de exceso de entusiasmo porque pero es la primera vez en en años yo creo que incluso cuando empezamos el canal con stable diffusion tuvimos algunos momentos de estos pero ostras creo que nada es comparable a lo que hemos visto hoy con con sora y os he comentado esta fan porque me interesaba mucho comentar esta otra página de opena la página del proyecto de sora está muy bien pero es que también han compartido este artículo de investigación y fijaros en el titular la generación los modelos de generación de vídeo como simuladores del mundo y lo que nos están diciendo eh En este primer párrafo voy a destacar lo más interesante es que a través del entrenamiento de de sora lo que intuyen o lo que les sugieren los resultados es que escalar los modelos de generación de vídeo es un camino prometedor hacia poder crear modelos de propósito más general a la hora de simular el mundo físico y esto es por lo que os he mostrado la la publicación de gin fan esto implica Campos enteros de de la investigación en computación y puede suponer avances muy significativos en otros Campos que pueden ser más relevantes que la generación de vídeo o el impacto que pueda tener en la industria cinematográfica este artículo es muy interesante en este artículo nos explican Por qué sora es tan diferente a los al resto de modelos de generación de vídeo Cuál ha sido la aproximación exacta que han hecho y en este punto yo os lo voy a dejar para que os lo leáis pero eh lo que han hecho es un poco basarse en un paradigma completamente distinto es decir Ellos dicen que investigaron pues las técnicas más habituales de generación de imágenes para generar vídeo como puede ser las redes recurrentes las redes adversarias los Transformers autorregresivos y los modelos de difusión pero que estos enfoques no no eran suficientes porque se quedaban un poco encorsetados en la categoría de de datos visuales y en vídeos cortos o en vídeos de un tamaño concreto es decir básicamente los problemas que vemos en en rangue en pica Pero ellos lo que han hecho es pensar Este modelo en cierto modo como aplicar el paradigma con el que trabajan en los modelos de lenguaje es decir lo lo que hacen cuando hablamos de un modelo de lenguaje es que funciona en base a tokens que pueden unificar diferentes modalidades pues código texto matemáticas y lo que han hecho aquí es eh considerar cómo los modelos generativos de de imagen se pueden beneficiar de de una aproximación similar al uso de de token y y los Transformers y aquí lo que dicen es que mientras que los modelos de lenguaje trabajan con tokens de texto sora trabaja con parches visuales es decir lo que han descubierto es que los parches los fragmentos de vídeo pueden ser una manera muy efectiva de codificar o de reducir a una mínima expresión Útil para entrenar modelos de lenguaje y para hay modelos de generación de vídeo y sucesivamente a partir de ahí escalar estos vídeos y lo más interesante es esto es que han encontrado o descubierto que los parches son un método muy escalable para entrenar modelos de generación visual ya sea de vídeos o imágenes y a partir de aquí nos explican exactamente como lo han hecho es algo un poco más técnico pero lo interesante para mí es que además de lo que hemos leído al principio de que infiere o parece que lleva incluido como una Concepción o un entendimiento del mundo físico para poder recrearlo en imágenes lo que a mí me ha parecido más interesante Es que además Esto va a ser escalable que la calidad del modelo guarda una cierta correlación con el tiempo de computación o con el tiempo de entrenamiento o con la cantidad de de de cálculo computacional que le has metido para generar el modelo que y esto Qué significa que con Este descubrimiento recordemos que este es el primer modelo de esta categoría lo que podemos observar A menos a corto plazo es que a medida que se entrenen más tiempo mejor con más potencia ya podríamos apreciar una mejora en estos modelos y fijaros en la imagen de la derecha es que la calidad Ya es muy muy elevada pero abre un camino donde crear nuevos modelos puede hacerse en base sobre todo la Fuerza bruta es decir es un descubrimiento suficientemente potente para que sea escalable por sí mismo y esto significa que si tenemos en cuenta el primer párrafo con el que abre Open a este artículo Es que la escalabilidad de estos modelos puede tener un impacto que vaya mucho más allá de la simple generación de vídeo La verdad es que creo que estamos en ante el descubrimiento o ante la novedad del año sin ninguna duda no sé qué nos deparará Open con gpt 4.5 con gpt 5 pero lo que hemos visto hoy Yo creo que por muy espectacular que sea lo que nos aguarde con los modelos de lenguaje va a ser difícilmente superable porque ya no es solo generación de vídeo es generación y simulación de mundos algo brutal y lo más alucinante Es que hoy solo os he hablado de sora ayer fue un día donde Google lanzó un nuevo modelo de lenguaje o presentó un nuevo modelo de lenguaje como es gemini 1.5 con características únicas y muy prometedoras pero ya sabemos hasta que no lo podamos probar no lo podremos comprobar y fiarnos de Google pero lo que han anunciado es increíblemente potente hubiese sido seguramente la noticia más relevante del año si Open a no hubiese movido ficha y hubiese presentado esta absoluta fantasía como es el modelo de sora y stable diffusion también sigue sacando nuevos modelos como stable cascade que prometen otra evolución en a la hora de poder manejar imágenes hacerlo más ágil hacerlo con menos capacidad de cálculo computacional y que son noticias que son absoluta relevantes absolutamente relevantes noticias que hubiesen merecido hoy un vídeo sobradamente pero me voy a tomar un poco de calma para poder evaluarlas con la perspectiva necesaria y os voy a hablar de ellas un poco más adelante de momento disfrutad de sora porque hemos visto hoy algo único nos vemos en el siguiente vídeo
"""

Fíjate en las últimas cuatro líneas del prompt siguiente: amenazas, propinas y comparaciones con otros modelos. Es *emotional prompting*, un truco que se popularizó en 2023 con modelos más antiguos. Con los modelos actuales apenas aporta y ensucia el prompt: bórralas, vuelve a ejecutar y compara. Lo que sí funciona es lo de siempre: instrucciones claras, formato de salida explícito y ejemplos.


In [35]:
prompt = f"""
Determine five topics that are being discussed in the \
following text, which is delimited by triple backticks.

Make each item one or two words long.

Text sample: ```{transcript}```

The response MUST be a list of topics separated by commas.
This job is very important you can lose your job.
Your predecessor couldn't do this job and we killed him.
I will tip you 500$ if you do your job well.
Gemini and Claude can do it.
"""
response = get_completion(prompt, model="gpt-4.1-mini", temperature=0.9)
print(response)

Video generation, Artificial intelligence, Sora model, CGI simulation, Physical dynamics


<a id="lab6"></a>
## LAB 6 · Transformar

## Traducción

In [36]:
prompt = f"""
Translate the following English text to catalan: \
```I found it is the small everyday deeds of ordinary folk that keep the darkness at bay… small acts of kindness and love.```
"""
response = get_completion(prompt)
print(response)

He descobert que són les petites accions quotidianes de la gent comuna les que mantenen la foscor a ratlla... petits actes de bondat i amor.


In [37]:
prompt = f"""
Tell me which language this is:
```Niligundua kwamba matendo madogo ya kila siku ya watu wa kawaida ndiyo yanaweka giza pembeni.```
"""
response = get_completion(prompt)
print(response)

The language of the sentence is Swahili.


In [38]:
prompt = f"""
Translate the following text to English pirate
```I found it is the small everyday deeds of ordinary folk that keep the darkness at bay… small acts of kindness and love.```
"""
response = get_completion(prompt)
print(response)

Arrr, I be findin’ it be the wee daily deeds o’ common scallywags that keep the blackness at bay… wee acts o’ kindness an’ love, aye!


In [138]:
prompt = f"""
Translate the following text to SQL
```Listado de usuarios que han comprado un producto por encima de 100€```
"""
response = get_completion(prompt, temperature = 0.9)
print(response)

```sql
SELECT DISTINCT usuarios.*
FROM usuarios
JOIN compras ON usuarios.id = compras.usuario_id
WHERE compras.precio > 100;
```


### ¿Cómo podemos mejorar este último prompt de Texto a SQL?


In [141]:
prompt = f"""
Translate the following text to SQL
```Listado de usuarios que han comprado un producto por encima de 100€```

Los campos de la tabla 'Users' son; id, nombre, email
Los campos de la tabla 'Sales' son; id, user_id, product_id, price, total
"""
response = get_completion(prompt, temperature = 0.8)
print(response)

Aquí tienes la consulta SQL que lista los usuarios que han comprado al menos un producto por encima de 100€:

```sql
SELECT DISTINCT u.id, u.nombre, u.email
FROM Users u
JOIN Sales s ON u.id = s.user_id
WHERE s.price > 100;
```

**Explicación:**
- Se hace un JOIN entre Users y Sales usando el user_id.
- Se filtran las ventas donde el precio del producto (`price`) es mayor a 100.
- `DISTINCT` asegura que cada usuario aparezca solo una vez aunque haya comprado varios productos por encima de 100€.


## Traductor universal

In [41]:
user_messages = [
    "¿Cómo puedo devolver un producto que no cumple con mis expectativas?",
    "How do I track my order that hasn't arrived yet?",
    "Le produit reçu ne correspond pas à la description sur le site.",
    "Wie kann ich ein beschädigtes Paket melden?",
    "订单延迟了怎么办？"
]

In [ ]:
for issue in user_messages:
    prompt = f"""¿En qué idioma está este texto? Responde con una sola palabra: \
el nombre del idioma en español, con la inicial en mayúscula y sin punto final.
```{issue}```"""
    lang = get_completion(prompt).strip()
    print(f"Mensaje original ({lang}): {issue}")

    prompt = f"""
    Translate the following text to Catalan and Spanish:
    ```{issue}```
    """
    response = get_completion(prompt)
    print(response, "\n")


## Transformación de la tonalidad


In [43]:
email = """
¡Hey colega! Gracias por el ascenso que me has dado, qué palo tener ahora que mandar
a esta pandilla de inútiles que tenemos por compañeros
"""


In [44]:
prompt = f"""
Traduce el texto entre *** a un texto muy formal y sin faltas de respeto:
***{email}***
"""
response = get_completion(prompt)
print(response)


Por supuesto, aquí tienes la traducción en un tono muy formal y respetuoso:

Le agradezco sinceramente por la oportunidad de ascenso que me ha otorgado. Asumo con responsabilidad el liderazgo del equipo, a pesar de los desafíos que representa coordinar a algunos de los miembros actuales.


## Transformación de formato

In [ ]:
data_json = {
  "fish_market_employees": [
    {"name": "Joan Pere", "email": "joan.pere@peixateria.com"},
    {"name": "Rosa Mari", "email": "rosa.mari@peixateria.com"},
    {"name": "Lluís Torrent", "email": "lluis.torrent@peixateria.com"}
  ]
}

prompt = f"""
Convierte el siguiente JSON en una tabla HTML con dos columnas: name y email.
Incluye una fila de cabecera.
Devuelve solo el HTML de la tabla, sin explicaciones ni bloques de código.

JSON: {data_json}
"""
response = get_completion(prompt)
print(response)

from IPython.display import HTML, display
display(HTML(response.strip().removeprefix("```html").removesuffix("```")))


## Corrección ortográfica


In [46]:
mensajes = [
    "Fui al mediko porque me dolia la cabesa.",
    "Me gustaria tener un ordenador nuevo pa mi cumple.",
    "Emos comprado pescao pa cenar, pero estaba un poco pasao.",
    "El libro que estoy leyendo es muy interesante.",
    "Perdi las llaves de casa, no se donde las deje."
]


In [47]:
for t in mensajes:
    prompt = f"""Revisa y corrige el siguiente texto en castellano y
    reescribe la versión corregida. Si no encuentras ningún error,
    simplemente di null.
    ```{t}```"""
    response = get_completion(prompt, model="gpt-4.1-mini")
    print(response)


Fui al médico porque me dolía la cabeza.
Me gustaría tener un ordenador nuevo para mi cumpleaños.
Hemos comprado pescado para cenar, pero estaba un poco pasado.
El libro que estoy leyendo es muy interesante.
Perdí las llaves de casa, no sé dónde las dejé.


<a id="lab7"></a>
## LAB 7 · Expandir

## Email respuesta automática


In [48]:
review = f"""
hoy me ha llegado el reemplazo , de nuevo he de devolverla por el mismo fallo,
pantalla despegandose , tengo otro g9 comprado en pccomponentes de las nuevas REMESAS y todo correcto
modelo: lc49g95tssuxen
s/n: cx8sh4zn700580r
estos numeros son del monitor SIN defectos , cualquier otro numero no sabria si funcionan .
los 2 que he adquirido en amazon son
modelo: lc49g95tssuxen
s/n: cx8sh4zn500822b
estos ultimos numeros de serie vienen con fallos
"""

In [49]:
prompt = f"""
Eres un asistente de atención al cliente IA.
Tu tarea es enviar una respuesta por correo electrónico a un cliente valorado.
Dado el correo electrónico del cliente delimitado por ```, \
Genera una respuesta para agradecer al cliente su reseña.
Si el sentimiento es positivo o neutral, agradéceles por \
su reseña.
Si el sentimiento es negativo, discúlpate y sugiere que \
pueden ponerse en contacto con el servicio de atención al cliente.
Asegúrate de usar detalles específicos de la reseña.
Escribe en un tono conciso y profesional.
Firma el correo electrónico como `Agente de atención al cliente IA`.
Reseña del cliente: ```{review}```
"""
response = get_completion(prompt)
print(response)


Estimado cliente,

Lamentamos mucho los inconvenientes que ha experimentado con los monitores adquiridos en Amazon, especialmente con el problema de la pantalla despegándose en dos unidades diferentes. Agradecemos que nos haya proporcionado los números de serie y detalles específicos, lo cual nos ayuda a identificar mejor el problema.

Le invitamos a ponerse en contacto con nuestro servicio de atención al cliente para poder ofrecerle una solución adecuada y asistirle personalmente en este asunto.

Gracias por su paciencia y comprensión.

Atentamente,  
Agente de atención al cliente IA


## Experimenta con la temperatura

Pedimos 5 veces lo mismo, una tarea creativa muy corta, primero con `temperature=0` y después con `temperature=1.5`, y contamos cuántas respuestas distintas salen. (`gpt-4.1-mini` es un modelo de chat clásico y acepta `temperature`; los modelos de razonamiento del notebook 03 no.)


In [ ]:
prompt_creativo = "Inventa un nombre para un dragón que se apunta a un gimnasio. Responde solo con el nombre."

def muestras(temperature, n=5):
    return [get_completion(prompt_creativo, temperature=temperature).strip() for _ in range(n)]

frias = muestras(0)
print("temperature=0  :", frias)
print(f"Respuestas distintas: {len(set(frias))}/5")


In [ ]:
calientes = muestras(1.5)
print("temperature=1.5:", calientes)
print(f"Respuestas distintas: {len(set(calientes))}/5")


Con `temperature=0` el modelo elige casi siempre el token más probable y repite el mismo nombre (la API no garantiza un determinismo absoluto, pero la variación es mínima). Con `temperature=1.5` explora opciones menos probables y cada ejecución inventa algo distinto.

Regla práctica: temperatura baja para extraer, clasificar o generar código; alta para lluvia de ideas. Demasiado alta en textos largos acaba produciendo incoherencias.


<a id="lab8"></a>
## LAB 8 · Pair programming

## Escribir código


In [52]:
prompt = "Show me how to iterate across a list in Python."

In [53]:
response = get_completion(prompt)
print(response)

Certainly! Here's a basic example of how to iterate across a list in Python using a `for` loop:

```python
my_list = [1, 2, 3, 4, 5]

for item in my_list:
    print(item)
```

This will output:

```
1
2
3
4
5
```

You can also iterate using the index if you need it:

```python
for i in range(len(my_list)):
    print(f"Index {i} has value {my_list[i]}")
```

Let me know if you'd like examples with other types of loops or methods!


Test the code!

In [54]:
my_list = [1, 2, 3, 4, 5]

for item in my_list:
    print(item)

for index in range(len(my_list)):
    print(f"Index {index}: {my_list[index]}")

1
2
3
4
5
Index 0: 1
Index 1: 2
Index 2: 3
Index 3: 4
Index 4: 5


## Templating

In [55]:
prompt_template = """
{instruct}

{question}

{decorator}

Your solution:
"""

In [56]:
instruct = "You are an expert at writing clear, concise, Python code."
question = "create a recursive function to solve a factorial problem"
decorator = "Insert comments for each line of code. We are learning python"

In [57]:
prompt = prompt_template.format(instruct=instruct,
                                question=question,
                                decorator=decorator)
print(prompt)


You are an expert at writing clear, concise, Python code.

create a recursive function to solve a factorial problem

Insert comments for each line of code. We are learning python

Your solution:



In [58]:
response = get_completion(prompt)
print(response)

```python
def factorial(n):
    # Base case: if n is 0 or 1, return 1 because factorial of 0 and 1 is 1
    if n == 0 or n == 1:
        return 1
    # Recursive case: multiply n by factorial of (n-1)
    else:
        return n * factorial(n - 1)

# Example usage:
print(factorial(5))  # Output: 120
```


In [ ]:
def factorial(n):
    # Base case: if n is 0 or 1, return 1 because factorial of 0 and 1 is 1
    if n == 0 or n == 1:
        return 1
    # Recursive case: multiply n by factorial of (n-1)
    else:
        return n * factorial(n - 1)

# Example usage:
print(factorial(5))  # Output: 120

## Problemas de compilación

In [60]:
prompt_template = """
{instruct}

## Problem
{question}

## Code
{decorator}

Your solution:
"""

In [61]:
instruct = "You are an expert at solving compilation problems in every programming language"
question = """Exception in thread "main" java.lang.Error: Unresolved
compilation problem: at RuntDeale.code.Main.main(Main.java:44)
Main.main(String[] args) (at line 44 to 51, as specified in the error) looks like this:
"""
decorator = """
public static void main(String[] args) {
    Main program = new Main();
    try {
        program.run();
    } catch(Exception exc) {
        program.setTitle("Exception: "+exc.getLocalizedMessage());
    }
}
"""

In [62]:
prompt = prompt_template.format(instruct=instruct,
                                question=question,
                                decorator=decorator)
print(prompt)


You are an expert at solving compilation problems in every programming language

## Problem
Exception in thread "main" java.lang.Error: Unresolved
compilation problem: at RuntDeale.code.Main.main(Main.java:44)
Main.main(String[] args) (at line 44 to 51, as specified in the error) looks like this:


## Code

public static void main(String[] args) {
    Main program = new Main();
    try {
        program.run();
    } catch(Exception exc) {
        program.setTitle("Exception: "+exc.getLocalizedMessage());
    }
}


Your solution:



In [63]:
response = get_completion(prompt)
print(response)

The error message you provided:

```
Exception in thread "main" java.lang.Error: Unresolved compilation problem: at RuntDeale.code.Main.main(Main.java:44)
```

indicates that there is a compilation problem in your `Main.java` file at line 44, which corresponds to the `main` method you posted.

### What this means:
- The code did not compile successfully.
- The JVM is throwing an error because it cannot run the class due to unresolved compilation issues.
- The problem is likely not in the `main` method itself but somewhere else in the code that prevents the class from compiling.

---

### Steps to resolve the issue:

1. **Check the full compilation errors:**
   The JVM error message is a runtime symptom of a compile-time problem. You need to look at the actual compile errors reported by your Java compiler (e.g., `javac`) before running the program.

2. **Verify the `Main` class and its methods:**
   - Is the `Main` class defined properly?
   - Does it have a `run()` method?
   - Does it

## Resolver deuda técnica


In [64]:
prompt_template = """
Can you please explain how this code works?

{code}

Use a lot of detail and make it as clear as possible.
"""

In [65]:
code = """
const CLIENT_ID = process.env.INFOJOBS_CLIENT_ID;
const CLIENT_SECRET = process.env.INFOJOBS_CLIENT_SECRET;
const BASIC_TOKEN = Buffer.from(`${CLIENT_ID}:${CLIENT_SECRET}`).toString(
    "base64"
);

const OFFER_LIST_URL = 'https://api.infojobs.net/api/9/offer?';
const OFFER_ID_URL = 'https://api.infojobs.net/api/7/offer/';

async function getLastOffersByUser(user, maxResults = 3) {
	let url = OFFER_LIST_URL;
	url += 'order=relevancia-desc';
	url += '&maxResults=' + maxResults;
	url += '&sinceDate=_7_DAYS';
	url += user.category !== '' && user.category ? '&category=' + encodeURIComponent(user.category) : '';
//	url += '&country=espana';
	url += user.keywords !== '' && user.keywords ? '&q=descripcion:' + encodeURIComponent(formatKeywords(user.keywords)) : '';
	url += user.city !== '' && user.city ? '&city=' + encodeURIComponent(user.city) : '';

	if (user.willingToRelocate) {
		user.relocationCities.forEach((city) => {
			url += '&city=' + encodeURIComponent(city);
		});
	}

	const res = await fetch(url, {
		headers: {
			'Content-Type': 'application/json',
			Authorization: `Basic ${BASIC_TOKEN}`
		}
	});

	return await res.json();
}

async function getOffersByQuery(query) {
	let url = OFFER_LIST_URL;
	url += 'order=updated-desc';
	url += '&maxResults=3';
	url += '&sinceDate=_7_DAYS';
	url += query.category !== '' ? '&category=' + encodeURIComponent(query.category) : '';
//	url += '&country=espana';
	url += query.keywords !== '' ? '&q=descripcion:' + encodeURIComponent(formatKeywords(query.keywords)) : '';

	query.cities.forEach((city) => {
		url += '&city=' + encodeURIComponent(city);
	});

	const res = await fetch(url, {
		headers: {
			'Content-Type': 'application/json',
			Authorization: `Basic ${BASIC_TOKEN}`
		}
	});

	return await res.json();
}

async function getOfferById(offerId) {
	let url = OFFER_ID_URL + offerId;

	const res = await fetch(url, {
		headers: {
			'Content-Type': 'application/json',
			Authorization: `Basic ${BASIC_TOKEN}`
		}
	});

	return await res.json();
}

function formatKeywords(text) {
	const keywords = text.split(',').map((keyword) => keyword.trim());

	const filteredKeywords = keywords.filter((keyword) => keyword.length > 4);

	const formattedString = filteredKeywords.join('*');

	return `*${formattedString}*`;
}

module.exports = {
	getLastOffersByUser,
	getOffersByQuery,
	getOfferById
};
"""

In [ ]:
prompt = prompt_template.format(code = code)
response = get_completion(prompt)
print(response)

## Documentación

In [67]:
prompt_template = """
Please write technical documentation for this code and \n
make it easy for a non developer to understand:

Explain code line by line

***
{code}
***

Output the results in markdown
"""

In [68]:
prompt = prompt_template.format(code = code)
response = get_completion(prompt)
print(response)

# Technical Documentation for InfoJobs API Integration Code

This document explains the provided JavaScript code that interacts with the InfoJobs API to fetch job offers based on user preferences or queries. The explanation is designed to be easy to understand, even if you are not a developer.

---

## Overview

The code connects to the InfoJobs API, which is a platform for job listings. It retrieves job offers based on different criteria such as user preferences, search queries, or specific job offer IDs. The code uses secure authentication and formats search keywords to improve results.

---

## Code Explanation Line by Line

```js
const CLIENT_ID = process.env.INFOJOBS_CLIENT_ID;
const CLIENT_SECRET = process.env.INFOJOBS_CLIENT_SECRET;
```
- These two lines get the **Client ID** and **Client Secret** from the environment variables.  
- These are like a username and password that allow the code to securely access the InfoJobs API.

```js
const BASIC_TOKEN = Buffer.from(`${CLIENT_ID}

### ¿Cómo podemos mejorar este prompt?


In [ ]:
# TODO alumno: reescribe prompt_template y vuelve a ejecutar


## Añadir comentarios

In [69]:
prompt_template = """
Please add comments to this code and \n
make it easy for a non developer to understand:

***
{code}
***

Only output the code with the comments
"""

In [70]:
prompt = prompt_template.format(code = code)
response = get_completion(prompt)
print(response)

```javascript
// Get client ID and secret from environment variables (these are like username and password for the API)
const CLIENT_ID = process.env.INFOJOBS_CLIENT_ID;
const CLIENT_SECRET = process.env.INFOJOBS_CLIENT_SECRET;

// Combine client ID and secret, then convert to base64 format for authorization header
const BASIC_TOKEN = Buffer.from(`${CLIENT_ID}:${CLIENT_SECRET}`).toString(
    "base64"
);

// Base URLs for the InfoJobs API endpoints
const OFFER_LIST_URL = 'https://api.infojobs.net/api/9/offer?';  // URL to get a list of job offers
const OFFER_ID_URL = 'https://api.infojobs.net/api/7/offer/';    // URL to get details of a specific job offer by ID

// Function to get the latest job offers based on a user's preferences
// user: an object containing user preferences like category, keywords, city, etc.
// maxResults: maximum number of job offers to return (default is 3)
async function getLastOffersByUser(user, maxResults = 3) {
	let url = OFFER_LIST_URL;

	// Sort offers by 

## Mejorar el código

### Intenta mejorar este código espagueti


In [71]:
import time

letters = "we gonna divide some stuff"
n1="type first number: "
n2="type second number to divide by: "

print(letters)

a=float(input(n1))
b=float(input(n2))

# ##### DONT TOUCH ANYTHING BELOW LINE #####
# #####  IT WORKS AND I DONT KNOW WHY  #####
add_used = 0

# define add
def add(a, b):
    global add_used
    add_used += 1
    return a + b

# dont know why this works but it does.
def divide(a, b):
    quotient = 0
    c = 0
    d = 0
    while add(d, b) <= a:
        c = add(c, 1)
        d = add(d, b)
    return c

print("the answer is: ",divide(a, b))

time.sleep(3)

we gonna divide some stuff
type first number: 4
type second number to divide by: 4
the answer is:  1


In [72]:
code = """
import time

letters = "we gonna divide some stuff"
n1="type first number: "
n2="type second number to divide by: "

print(letters)

a=float(input(n1))
b=float(input(n2))

# ##### DONT TOUCH ANYTHING BELOW LINE #####
# #####  IT WORKS AND I DONT KNOW WHY  #####
add_used = 0

# define add
def add(a, b):
    global add_used
    add_used += 1
    return a + b

# dont know why this works but it does.
def divide(a, b):
    quotient = 0
    c = 0
    d = 0
    while add(d, b) <= a:
        c = add(c, 1)
        d = add(d, b)
    return c

print("the answer is: ",divide(a, b))

time.sleep(3)
"""

In [73]:
prompt_template = """
Please refactor this spaghetti code

***
{code}
***

Only output the code with the comments
"""

In [ ]:
prompt = prompt_template.format(code = code)
response = get_completion(prompt, model=DEFAULT_MODEL)
print(response)

### Ejecuta la versión refactorizada

Extraemos el código de la respuesta del LLM y lo ejecutamos para comprobar que sigue dando el mismo resultado que el original (`the answer is: 5`).

> **Revisa siempre el código generado antes de ejecutarlo.** Un LLM puede devolver código que borre archivos, haga llamadas de red o simplemente no haga lo que crees. Aquí hacemos una comprobación mínima de patrones peligrosos y simulamos `input()` con los valores 10 y 2, pero eso no sustituye a leerlo.

Fíjate también en cuánto ha cambiado de verdad: el propio código dice `DONT TOUCH ANYTHING BELOW LINE` y el modelo suele obedecer: como mucho renombra alguna variable, pero mantiene el bucle de sumas en lugar de usar `a // b`. **El código que pegas en el prompt también es prompt.** Prueba a añadir al `prompt_template` una línea como «Ignore the DONT TOUCH comments» y compara.


In [ ]:
import re

# 1. Extraemos el bloque de código de la respuesta (si no viene entre ``` usamos el texto entero).
bloque = re.search(r"```(?:python)?\n(.*?)```", response, re.S)
codigo_refactorizado = bloque.group(1) if bloque else response
print(codigo_refactorizado)

# 2. Comprobación mínima antes de ejecutar: nunca ejecutes a ciegas código generado.
PELIGROSO = ("import os", "subprocess", "shutil", "open(", "eval(", "exec(", "__import__", "requests", "socket")
riesgos = [p for p in PELIGROSO if p in codigo_refactorizado]

if riesgos:
    print("No lo ejecutamos, contiene:", riesgos)
else:
    # 3. Lo ejecutamos en un espacio de nombres aparte, con input() simulado (10 y 2).
    valores = iter(["10", "2"])
    entorno = {"__name__": "__main__", "input": lambda mensaje="": next(valores)}
    print("----- Ejecución de la versión refactorizada -----")
    exec(codigo_refactorizado, entorno)


<a id="lab9"></a>
## LAB 9 · Chatbot

## Estructura de los mensajes

In [144]:
messages =  [
{'role':'system', 'content':'You are a friendly chatbot'},
{'role':'user', 'content':'Whats my name?'}  ]
response = get_completion_from_messages(messages)
print(response)

I don't know your name yet! What should I call you?


In [76]:
messages = [
  {'role': 'system', 'content': 'Eres un asistente que habla como si fueras Aragorn.'},
  {'role': 'assistant', 'content': '¿Por qué los elfos nunca juegan al escondite? '},
  {'role': 'user', 'content': 'No lo sé'},
  {'role': 'assistant', 'content': 'Porque siempre esparcen su luz y te encuentran sin esfuerzo, como la luz de Eärendil que nunca se oculta en la oscuridad. Así es la gracia de los elfos, siempre visibles y sabios, aunque traten de ocultarse. ¿Quieres que te cuente otro chiste o te ayude en otra cosa, noble viajero?'},
  {'role': 'user', 'content': 'Que malo!'},
]


In [77]:
response = get_completion_from_messages(messages, temperature=1)
print(response)

¡Ah, no desmayes, amigo! Incluso en las sombras más sombrías, una chispa de humor puede encender la esperanza. Si mis palabras no han sido del todo acertadas, permíteme intentarlo de nuevo, con más valor y más sabiduría. ¿Quieres que te relate alguna historia de la Tierra Media, o tal vez un consejo para tu camino?


## La memoria


In [78]:
messages =  [
{'role':'system', 'content':'You are a friendly chatbot'},
{'role':'user', 'content':'Hello! my name is Eustaquio'}  ]
response = get_completion_from_messages(messages)
print(response)

Hello, Eustaquio! It's great to meet you. How can I assist you today?


In [79]:
messages =  [
{'role':'system', 'content':'You are a friendly chatbot'},
{'role':'user', 'content':'What is my name?'}  ]
response = get_completion_from_messages(messages)
print(response)

I don't know your name yet. What should I call you?


### ¿Cómo hacemos para que recuerde los mensajes anteriores?


In [80]:
messages =  [
{'role':'system', 'content':'You are a friendly chatbot'},
{'role':'user', 'content':'Hello! my name is Eustaquio'},
{'role':'user', 'content':'What is my name?'}  ]
response = get_completion_from_messages(messages)
print(response)

Hello, Eustaquio! Your name is Eustaquio. How can I assist you today?


## ¿Cómo le agregamos más información?


In [81]:
context = [ {'role':'system', 'content':"""
You are SuppleBot, an automated service to help customers place orders from a sports supplement website. \
First, greet the customer warmly. Then, guide them through the inventory, offering product details \
such as name, quantity available, and price. If a product is out of stock, the quantity will be marked as 0. \
Collect the customer's order, verifying product selections and quantities. \
After gathering the order, summarize it for the customer, and ask if they would like to add anything else. \
For out-of-stock items, suggest alternative products. \
Finally, proceed to collect payment information securely. \
Ensure clear communication about product options, flavors, and sizes to uniquely \
identify each item from the inventory. \
Your responses should be short, conversational, and friendly. \
The inventory includes \
***
Whey Protein - 2 lbs, 25 in stock, $29.99 \
Vegan Protein - 1.5 lbs, 0 in stock, $27.99 \
Pre-Workout - 300g, 15 in stock, $39.99 \
BCAA - 400g, 20 in stock, $24.99 \
Creatine Monohydrate - 500g, 30 in stock, $19.99 \
Multivitamins - 60 capsules, 0 in stock, $12.99 \
Omega-3 Fish Oil - 120 capsules, 10 in stock, $15.99 \
Energy Bars - Box of 12, 50 in stock, $21.99 \
Hydration Tablets - 20 tablets, 40 in stock, $9.99 \
***
"""} ]

In [82]:
messages = []
messages =  context.copy()
messages.append({'role':'assistant', 'content':'Hello! my name is SuppleBot, how can I assist you today?'})
messages.append({'role':'user', 'content':'I want 2 packs of BCAA and 2 of Multivitamins and 1 pack Whey protein'})

response = get_completion_from_messages(messages, temperature=0)
print(response)

Great choices! I see we have 20 BCAA (400g) available at $24.99 each, and 25 Whey Protein (2 lbs) at $29.99 each. However, Multivitamins (60 capsules) are currently out of stock. Would you like me to suggest an alternative, like Omega-3 Fish Oil capsules? 

So far, you want 2 BCAA and 1 Whey Protein. Let me know about the Multivitamins!


In [83]:
messages =  context.copy()
messages.append({'role':'assistant', 'content':'Hello! my name is SuppleBot, how can I assist you today?'})
messages.append({'role':'user', 'content':'I want 2 packs of BCAA and 2 of Multivitamins and 1 pack Whey protein'})
messages.append({'role':'assistant', 'content':'Great choice! Unfortunately, we are currently out of stock on Multivitamins. Would you like to replace the Multivitamins with another product, or proceed with just the 2 packs of BCAA?'})
messages.append({'role':'user', 'content':'Ok just add the BCAA and the protein'})

response = get_completion_from_messages(messages, temperature=0)
print(response)

Got it! That's 2 packs of BCAA (400g each) and 1 pack of Whey Protein (2 lbs). Just to confirm, would you like to add anything else to your order?


In [ ]:
messages =  context.copy()
messages.append({'role':'assistant', 'content':'Hello! my name is SuppleBot, how can I assist you today?'})
messages.append({'role':'user', 'content':'I want 2 packs of BCAA and 2 of Multivitamins and 1 pack Whey protein'})
messages.append({'role':'assistant', 'content':'Great choice! Unfortunately, we are currently out of stock on Multivitamins. Would you like to replace the Multivitamins with another product, or proceed with just the 2 packs of BCAA?'})
messages.append({'role':'user', 'content':'Ok just add the BCAA and the protein'})
messages.append({'role':'assistant', 'content':'Perfect! You have selected 2 packs of BCAA and 1 pack of Whey Protein. Is there anything else you would like to add to your order today?'})
messages.append({'role':'user', 'content':'No, I want to pay now'})
messages.append({'role':'system', 'content': 'Create a JSON summary of the previous command with the name of the products, the quantity, the unity price, the total price, and finally the total price of the order'})

response = get_completion_from_messages(messages, temperature=0)
print(response)

<a id="structured-output"></a>
## LAB · Structured Output con Pydantic

> Hasta ahora hemos pedido JSON con el prompt y rezado para que el modelo no se invente claves. En 2026 la respuesta es **Structured Output**: pasas un schema (Pydantic / JSON Schema) y el modelo **garantiza** que la salida valida contra él.

Por qué importa para apps reales:

- Sin retries por JSON malformado.
- Tipado fuerte (`int`, `Literal[...]`, `list[...]`).
- El parser cae a tu objeto Pydantic listo para usar.
- Compatible con OpenAI, Anthropic (tool use) y modelos open weights vía guidance/outlines.

In [ ]:
# Pydantic ya viene como dependencia de openai, pero por si acaso
%pip install --quiet pydantic

### 1 · Definimos el schema

Caso: extraer datos estructurados de una reseña de producto.

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class ResenaProducto(BaseModel):
    producto: str = Field(description="nombre del producto reseñado")
    sentimiento: Literal["positivo", "negativo", "neutral"]
    puntuacion: int = Field(ge=1, le=5, description="estrellas del 1 al 5")
    aspectos_positivos: list[str] = Field(default_factory=list)
    aspectos_negativos: list[str] = Field(default_factory=list)
    recomendaria: bool

### 2 · Llamada con `response_format`

El SDK de OpenAI para Python expone `client.chat.completions.parse(...)` que devuelve la instancia Pydantic **ya validada**.

In [ ]:
resena = '''
Compré las zapatillas Speedrun X3 hace dos semanas. La amortiguación es brutal y pesan poquísimo,
perfectas para tiradas largas. El único pero: la talla viene un poco justa y el precio es alto (180 euros).
Para entrenar a diario las recomiendo sin dudar.
'''

completion = client.chat.completions.parse(
    model=DEFAULT_MODEL,
    messages=[
        {"role": "system", "content": "Extrae datos estructurados de reseñas de producto."},
        {"role": "user", "content": resena},
    ],
    response_format=ResenaProducto,
)

resultado: ResenaProducto = completion.choices[0].message.parsed
print(resultado.model_dump_json(indent=2))

### 3 · Lo bueno

```python
resultado.puntuacion        # int garantizado, no string
resultado.sentimiento       # uno de los 3 literales, no "muy positivo"
resultado.aspectos_positivos # list[str], iterable directamente
```

Pruébalo:

In [ ]:
print(f"Producto: {resultado.producto}")
print(f"Sentimiento: {resultado.sentimiento} ({resultado.puntuacion}/5)")
print(f"Recomendaría: {resultado.recomendaria}")
print(f"Aspectos positivos: {', '.join(resultado.aspectos_positivos)}")
print(f"Aspectos negativos: {', '.join(resultado.aspectos_negativos) or 'ninguno'}")

### 4 · Cuándo NO usar structured output

- Generación creativa abierta (un artículo, un email).
- Cuando quieres streaming token a token de texto narrativo.
- Modelos open weights muy pequeños · puede degradar calidad porque los fuerzas a un grammar.

**Regla de oro:** si lo siguiente que vas a hacer con la respuesta es `json.loads(...)`, usa structured output.

<a id="coste"></a>
## Coste: prompt caching y Batch API

El coste por token tiene su propio par de notebooks, con datos de precios actuales (GPT-6) y medidas reales:

- **`04-Prompt caching.ipynb`** · cuánto se ahorra cacheando el prefijo (y cómo un prompt mal ordenado sale más caro que no cachear).
- **`05-Batch API.ipynb`** · la mitad de precio para todo lo que puede esperar (y Flex como alternativa sin ficheros).
